In [ ]:
import pandas as pd

df = pd.read_csv("/content/Cyclone_Geospatial_Features.csv")

print(df.head())
print(df.shape)
print(df.columns)
print(df.isnull().sum())
# Remove Earth Engine metadata
df = df.drop(columns=["system:index", ".geo"])

# Features we'll use for ML
X = df[
    [
        "elevation",
        "slope",
        "rainfall",
        "distance_to_coast",
        "population"
    ]
]

print("Dataset shape:", X.shape)
print("\nFeatures:")
print(X.columns.tolist())
print("\nFirst 5 rows:")
print(X.head())

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

print(X_scaled.head())


# Create a simple baseline vulnerability score
baseline_score = (
    0.25 * X_scaled["rainfall"] +
    0.25 * X_scaled["population"] +
    0.20 * (1 - X_scaled["elevation"]) +
    0.20 * (1 - X_scaled["distance_to_coast"]) +
    0.10 * (1 - X_scaled["slope"])
)

df["baseline_vulnerability"] = baseline_score

print(df[[
    "elevation",
    "slope",
    "rainfall",
    "distance_to_coast",
    "population",
    "baseline_vulnerability"
]].head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/Cyclone_Geospatial_Features.csv'

In [ ]:
df.to_csv("cyclone_baseline_features.csv", index=False)

print("Saved:", "cyclone_baseline_features.csv")
print("Rows:", len(df))

In [ ]:
import pandas as pd

# See uploaded files
import os
print(os.listdir("/content"))

In [ ]:
import pandas as pd

track = pd.read_csv(
    "/content/ibtracs.NI.list.v04r01.csv",
    low_memory=False
)

print("Shape:", track.shape)
print("\nColumns:")
print(track.columns.tolist())

In [ ]:
print(track.head())

In [ ]:
print(track[
    ["SID", "SEASON", "NAME", "ISO_TIME", "LAT", "LON", "WMO_WIND", "WMO_PRES"]
].head(10))

In [ ]:
# Remove the units row (row 0)
track = track.iloc[1:].copy()

# Keep only the fields we need
track = track[
    [
        "SID",
        "SEASON",
        "BASIN",
        "SUBBASIN",
        "NAME",
        "ISO_TIME",
        "LAT",
        "LON",
        "WMO_WIND",
        "WMO_PRES",
        "DIST2LAND",
        "LANDFALL"
    ]
].copy()

# Convert numeric columns
numeric_cols = [
    "SEASON",
    "LAT",
    "LON",
    "WMO_WIND",
    "WMO_PRES",
    "DIST2LAND"
]

for col in numeric_cols:
    track[col] = pd.to_numeric(track[col], errors="coerce")

# Convert time
track["ISO_TIME"] = pd.to_datetime(
    track["ISO_TIME"],
    errors="coerce"
)

print("Shape:", track.shape)
print(track.head())
print("\nMissing values:")
print(track.isnull().sum())

In [ ]:
print(
    "Year range:",
    track["SEASON"].min(),
    "to",
    track["SEASON"].max()
)

print("\nNumber of unique cyclones:")
print(track["SID"].nunique())


In [ ]:
recent_bb = track[
    (track["SUBBASIN"] == "BB") &
    (track["SEASON"] >= 2000)
]

print("Recent Bay of Bengal track points:", len(recent_bb))

print(
    recent_bb[
        ["SID", "SEASON", "NAME", "ISO_TIME", "LAT", "LON", "WMO_WIND"]
    ]
    .drop_duplicates("SID")
    .sort_values("SEASON")
    .tail(30)
)


In [ ]:
# Expanded region around our study area
regional_tracks = track[
    (track["SUBBASIN"] == "BB") &
    (track["LON"].between(86.0, 90.0)) &
    (track["LAT"].between(19.0, 24.5))
].copy()

print("Track points near study region:", len(regional_tracks))

# List unique cyclones
regional_storms = (
    regional_tracks[
        ["SID", "SEASON", "NAME"]
    ]
    .drop_duplicates("SID")
    .sort_values(["SEASON", "NAME"])
)

print("Unique relevant cyclone systems:", len(regional_storms))
print(regional_storms.to_string(index=False))


In [ ]:
# Calculate approximate distance from each track point
# to the CENTER of our current study area
center_lat = (21.5 + 22.5) / 2
center_lon = (87.8 + 88.5) / 2

regional_tracks["distance_from_center"] = (
    ((regional_tracks["LAT"] - center_lat) ** 2 +
     (regional_tracks["LON"] - center_lon) ** 2) ** 0.5
)

closest_points = (
    regional_tracks
    .sort_values("distance_from_center")
    .groupby("SID")
    .first()
    .reset_index()
    [["SID", "SEASON", "NAME", "LAT", "LON", "WMO_WIND",
      "WMO_PRES", "distance_from_center"]]
    .sort_values("distance_from_center")
)

print(closest_points.head(20).to_string(index=False))


In [ ]:
# Cyclones that passed close to our study region
candidate_storms = (
    closest_points[
        closest_points["distance_from_center"] <= 1.0
    ]
    .sort_values(["SEASON", "distance_from_center"])
)

print(
    candidate_storms[
        ["SID", "SEASON", "NAME", "LAT", "LON",
         "WMO_WIND", "WMO_PRES", "distance_from_center"]
    ].to_string(index=False)
)

In [ ]:
# Cyclones that passed close to our study region
candidate_storms = (
    closest_points[
        closest_points["distance_from_center"] <= 1.0
    ]
    .sort_values(["SEASON", "distance_from_center"])
)

print(
    candidate_storms[
        ["SID", "SEASON", "NAME", "LAT", "LON",
         "WMO_WIND", "WMO_PRES", "distance_from_center"]
    ].to_string(index=False)
)

In [ ]:
# Focus on modern events with satellite coverage
modern_candidates = candidate_storms[
    candidate_storms["SEASON"] >= 2014
]

print(
    modern_candidates[
        ["SID", "SEASON", "NAME", "LAT", "LON",
         "WMO_WIND", "WMO_PRES", "distance_from_center"]
    ].to_string(index=False)
)

In [ ]:
import numpy as np

# Study-area center
center_lat = 22.0
center_lon = 88.15

# Haversine distance in km
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


regional_tracks["distance_km"] = haversine(
    center_lat,
    center_lon,
    regional_tracks["LAT"],
    regional_tracks["LON"]
)

# Find closest point for every cyclone
closest_storm_points = (
    regional_tracks
    .loc[
        regional_tracks.groupby("SID")["distance_km"].idxmin()
    ]
    [
        [
            "SID",
            "SEASON",
            "NAME",
            "LAT",
            "LON",
            "WMO_WIND",
            "WMO_PRES",
            "distance_km"
        ]
    ]
    .sort_values("distance_km")
)

print(closest_storm_points.head(30).to_string(index=False))

In [ ]:
modern_storms = closest_storm_points[
    (closest_storm_points["SEASON"] >= 2014) &
    (closest_storm_points["distance_km"] <= 200)
].copy()

print(
    modern_storms.to_string(index=False)
)

print(
    "\nNumber of candidate modern storms:",
    len(modern_storms)
)

In [ ]:
amphan = track[
    track["SID"] == "2020136N10088"
].copy()

print("Amphan track:")
print(
    amphan[
        ["ISO_TIME", "LAT", "LON", "WMO_WIND", "WMO_PRES", "DIST2LAND"]
    ].to_string(index=False)
)

print("\nStart:", amphan["ISO_TIME"].min())
print("End:", amphan["ISO_TIME"].max())


In [ ]:
import pandas as pd

amphan_df = pd.read_csv(
    "/content/Amphan_ML_Dataset.csv"
)

print(amphan_df.shape)
print(amphan_df.columns)
print(amphan_df.head())
print("\nLabel distribution:")
print(amphan_df["flood_label"].value_counts())

In [ ]:
# Remove Earth Engine metadata
amphan_df = amphan_df.drop(
    columns=["system:index", ".geo"]
)

# Features
X = amphan_df[
    [
        "elevation",
        "slope",
        "rainfall",
        "distance_to_coast",
        "population"
    ]
]

# Target
y = amphan_df["flood_label"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:")
print(X.head())
print("\nTarget:")
print(y.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Random Forest trained successfully.")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Feature Importance:")
print(importance)


In [ ]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", auc)


In [ ]:
import joblib

joblib.dump(model, "/content/amphan_random_forest.pkl")

print("Model saved successfully.")

In [ ]:
amphan_df.to_csv(
    "/content/amphan_training_data.csv",
    index=False
)

print("Dataset saved:", amphan_df.shape)

In [ ]:
import pandas as pd

amphan = pd.read_csv("/content/Amphan_ML_Dataset.csv")
fani = pd.read_csv("/content/Fani_ML_Dataset.csv")
yaas = pd.read_csv("/content/Yaas_ML_Dataset.csv")
remal = pd.read_csv("/content/Remal_ML_Dataset.csv")

# Add event identity
amphan["event"] = "Amphan"
fani["event"] = "Fani"
yaas["event"] = "Yaas"
remal["event"] = "Remal"

# Combine
all_events = pd.concat(
    [amphan, fani, yaas, remal],
    ignore_index=True
)

print("Combined dataset:", all_events.shape)
print("\nEvents:")
print(all_events["event"].value_counts())

print("\nLabels:")
print(all_events["flood_label"].value_counts())

In [ ]:
# Remove columns we don't need for ML
ml_data = all_events.drop(
    columns=["system:index", ".geo"]
)

# Keep event separately for event-based testing
events = ml_data["event"]

# Features
X = ml_data[
    [
        "elevation",
        "slope",
        "rainfall",
        "distance_to_coast",
        "population"
    ]
]

# Target
y = ml_data["flood_label"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())

In [ ]:
# Train on 3 cyclones, test on the 4th
train_mask = events != "Remal"
test_mask = events == "Remal"

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining events:")
print(events[train_mask].value_counts())

print("\nTesting events:")
print(events[test_mask].value_counts())

print("\nTest labels:")
print(y_test.value_counts())

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_multi = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model_multi.fit(X_train, y_train)

print("Multi-cyclone Random Forest trained!")


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

y_pred = model_multi.predict(X_test)
y_prob = model_multi.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
import os

files = os.listdir("/content")

for f in files:
    print(f)


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
import pandas as pd

amphan = pd.read_csv("/content/Amphan_ML_Dataset.csv")
fani = pd.read_csv("/content/Fani_ML_Dataset.csv")
yaas = pd.read_csv("/content/Yaas_ML_Dataset.csv")
remal = pd.read_csv("/content/Remal_ML_Dataset.csv")

print("Amphan:", amphan.shape)
print("Fani:", fani.shape)
print("Yaas:", yaas.shape)
print("Remal:", remal.shape)

In [ ]:
import json

def extract_coordinates(df):
    df = df.copy()

    def get_lon_lat(geo):
        point = json.loads(geo)
        lon, lat = point["coordinates"]
        return pd.Series([lon, lat])

    df[["lon", "lat"]] = df[".geo"].apply(get_lon_lat)

    return df


amphan = extract_coordinates(amphan)
fani = extract_coordinates(fani)
yaas = extract_coordinates(yaas)
remal = extract_coordinates(remal)

print(amphan[["lat", "lon"]].head())

In [ ]:
track = pd.read_csv(
    "/content/ibtracs.NI.list.v04r01.csv",
    low_memory=False
)

# Remove the units row
track = track.iloc[1:].copy()

# Keep only what we need
track = track[
    [
        "SID",
        "SEASON",
        "BASIN",
        "SUBBASIN",
        "NAME",
        "ISO_TIME",
        "LAT",
        "LON",
        "WMO_WIND",
        "WMO_PRES",
        "DIST2LAND",
        "LANDFALL"
    ]
].copy()

# Convert numeric columns
numeric_cols = [
    "SEASON",
    "LAT",
    "LON",
    "WMO_WIND",
    "WMO_PRES",
    "DIST2LAND"
]

for col in numeric_cols:
    track[col] = pd.to_numeric(track[col], errors="coerce")

track["ISO_TIME"] = pd.to_datetime(
    track["ISO_TIME"],
    errors="coerce"
)

print("Track dataset:", track.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/ibtracs.NI.list.v04r01.csv'

In [ ]:
storms = (
    track[
        (track["SUBBASIN"] == "BB") &
        (track["NAME"].isin(["AMPHAN", "FANI", "YAAS", "REMAL"]))
    ]
    .groupby(["NAME", "SEASON", "SID"])
    .size()
    .reset_index(name="track_points")
)

print(storms.to_string(index=False))

In [ ]:
storm_sids = {
    "Amphan": "2020136N10088",
    "Fani": "2019116N02090",
    "Yaas": "2021143N15090",
    "Remal": "2024145N14087"
}

tracks = {}

for event, sid in storm_sids.items():
    tracks[event] = (
        track[track["SID"] == sid]
        .dropna(subset=["LAT", "LON"])
        .copy()
        .sort_values("ISO_TIME")
    )

    print(
        event,
        "→",
        len(tracks[event]),
        "track points |",
        tracks[event]["ISO_TIME"].min(),
        "to",
        tracks[event]["ISO_TIME"].max()
    )

In [ ]:
import numpy as np

def haversine_matrix(
    sample_lat,
    sample_lon,
    track_lat,
    track_lon
):
    R = 6371.0

    sample_lat = np.radians(sample_lat[:, None])
    sample_lon = np.radians(sample_lon[:, None])

    track_lat = np.radians(track_lat[None, :])
    track_lon = np.radians(track_lon[None, :])

    dlat = track_lat - sample_lat
    dlon = track_lon - sample_lon

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(sample_lat)
        * np.cos(track_lat)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


def add_track_features(samples, cyclone_track):
    samples = samples.copy()

    sample_lat = samples["lat"].to_numpy()
    sample_lon = samples["lon"].to_numpy()

    track_lat = cyclone_track["LAT"].to_numpy()
    track_lon = cyclone_track["LON"].to_numpy()

    distances = haversine_matrix(
        sample_lat,
        sample_lon,
        track_lat,
        track_lon
    )

    nearest_idx = np.argmin(distances, axis=1)

    samples["track_distance_km"] = (
        distances[
            np.arange(len(samples)),
            nearest_idx
        ]
    )

    samples["track_wind_kt"] = (
        cyclone_track["WMO_WIND"]
        .to_numpy()[nearest_idx]
    )

    samples["track_pressure_hpa"] = (
        cyclone_track["WMO_PRES"]
        .to_numpy()[nearest_idx]
    )

    return samples

In [ ]:
amphan = add_track_features(amphan, tracks["Amphan"])
fani = add_track_features(fani, tracks["Fani"])
yaas = add_track_features(yaas, tracks["Yaas"])
remal = add_track_features(remal, tracks["Remal"])

print(
    amphan[
        [
            "lat",
            "lon",
            "track_distance_km",
            "track_wind_kt",
            "track_pressure_hpa"
        ]
    ].head()
)

In [ ]:
# Add event names
amphan["event"] = "Amphan"
fani["event"] = "Fani"
yaas["event"] = "Yaas"
remal["event"] = "Remal"

all_events = pd.concat(
    [amphan, fani, yaas, remal],
    ignore_index=True
)

print("Combined dataset:", all_events.shape)
print("\nEvents:")
print(all_events["event"].value_counts())

In [ ]:
ml_data = all_events.drop(
    columns=["system:index", ".geo"]
)

X = ml_data[
    [
        "elevation",
        "slope",
        "rainfall",
        "distance_to_coast",
        "population",
        "track_distance_km",
        "track_wind_kt",
        "track_pressure_hpa"
    ]
]

y = ml_data["flood_label"]
events = ml_data["event"]

print("X shape:", X.shape)
print("Features:")
print(X.columns.tolist())

NameError: name 'all_events' is not defined

In [ ]:
train_mask = events != "Remal"
test_mask = events == "Remal"

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_track = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model_track.fit(X_train, y_train)

print("Track-aware Random Forest trained!")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

y_pred_track = model_track.predict(X_test)
y_prob_track = model_track.predict_proba(X_test)[:, 1]

print("TRACK-AWARE MODEL")
print("=" * 40)

print("Accuracy:", accuracy_score(y_test, y_pred_track))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_track))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_track))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_track))

NameError: name 'model_track' is not defined

In [ ]:
print("MODEL COMPARISON")
print("=" * 40)

print("Previous model:")
print("  Accuracy:", accuracy_score(y_test, y_pred))
print("  ROC-AUC :", roc_auc_score(y_test, y_prob))

print("\nTrack-aware model:")
print("  Accuracy:", accuracy_score(y_test, y_pred_track))
print("  ROC-AUC :", roc_auc_score(y_test, y_prob_track))

In [ ]:
for name, df in {
    "Amphan": amphan,
    "Fani": fani,
    "Yaas": yaas,
    "Remal": remal
}.items():

    print(f"\n{name}")
    print(df[
        [
            "track_distance_km",
            "track_wind_kt",
            "track_pressure_hpa"
        ]
    ].describe())

In [ ]:
print(
    all_events.groupby("flood_label")[
        [
            "track_distance_km",
            "track_wind_kt",
            "track_pressure_hpa"
        ]
    ].mean()
)

In [ ]:
for name, df in tracks.items():

    print(f"\n{name}")
    print(
        "Track latitude:",
        df["LAT"].min(),
        "→",
        df["LAT"].max()
    )
    print(
        "Track longitude:",
        df["LON"].min(),
        "→",
        df["LON"].max()
    )

print("\nStudy sample coordinates:")

combined_coords = pd.concat(
    [amphan, fani, yaas, remal]
)

print(
    "Latitude:",
    combined_coords["lat"].min(),
    "→",
    combined_coords["lat"].max()
)

print(
    "Longitude:",
    combined_coords["lon"].min(),
    "→",
    combined_coords["lon"].max()
)

In [ ]:
for name, df in {
    "Amphan": amphan,
    "Fani": fani,
    "Yaas": yaas,
    "Remal": remal
}.items():

    print(
        f"{name}: "
        f"closest sample-track distance = "
        f"{df['track_distance_km'].min():.2f} km"
    )

In [ ]:
def add_hazard_features(samples, cyclone_track):
    samples = samples.copy()

    sample_lat = samples["lat"].to_numpy()
    sample_lon = samples["lon"].to_numpy()

    track_lat = cyclone_track["LAT"].to_numpy()
    track_lon = cyclone_track["LON"].to_numpy()

    distances = haversine_matrix(
        sample_lat,
        sample_lon,
        track_lat,
        track_lon
    )

    # Distance-weighted wind hazard
    winds = cyclone_track["WMO_WIND"].to_numpy()
    winds = np.nan_to_num(winds, nan=0)

    # Stronger nearby winds contribute more
    wind_hazard = winds[None, :] / (distances + 10)

    samples["wind_distance_hazard"] = np.max(
        wind_hazard,
        axis=1
    )

    # Distance to the strongest/intense portion of the track
    samples["min_track_distance_km"] = np.min(
        distances,
        axis=1
    )

    return samples

In [ ]:
amphan = add_hazard_features(
    amphan, tracks["Amphan"]
)

fani = add_hazard_features(
    fani, tracks["Fani"]
)

yaas = add_hazard_features(
    yaas, tracks["Yaas"]
)

remal = add_hazard_features(
    remal, tracks["Remal"]
)

print(
    amphan[
        [
            "lat",
            "lon",
            "min_track_distance_km",
            "wind_distance_hazard"
        ]
    ].head()
)

In [ ]:
all_events_hazard = pd.concat(
    [amphan, fani, yaas, remal],
    ignore_index=True
)

ml_data_hazard = all_events_hazard.drop(
    columns=["system:index", ".geo"]
)

X_hazard = ml_data_hazard[
    [
        "elevation",
        "slope",
        "rainfall",
        "distance_to_coast",
        "population",
        "min_track_distance_km",
        "wind_distance_hazard"
    ]
]

y_hazard = ml_data_hazard["flood_label"]
events_hazard = ml_data_hazard["event"]

print("Dataset:", X_hazard.shape)
print("\nFeatures:")
print(X_hazard.columns.tolist())

In [ ]:
train_mask = events_hazard != "Remal"
test_mask = events_hazard == "Remal"

X_train_h = X_hazard[train_mask]
y_train_h = y_hazard[train_mask]

X_test_h = X_hazard[test_mask]
y_test_h = y_hazard[test_mask]

model_hazard = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model_hazard.fit(X_train_h, y_train_h)

print("Hazard-aware model trained!")

In [ ]:
y_pred_h = model_hazard.predict(X_test_h)
y_prob_h = model_hazard.predict_proba(X_test_h)[:, 1]

print("HAZARD-AWARE MODEL")
print("=" * 40)

print("Accuracy:", accuracy_score(y_test_h, y_pred_h))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_h, y_pred_h))

print("\nClassification Report:")
print(
    classification_report(
        y_test_h,
        y_pred_h,
        zero_division=0
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(y_test_h, y_prob_h)
)

In [ ]:
print("Flood rate by event:")
print(
    all_events_hazard.groupby("event")["flood_label"]
    .mean()
    .sort_values(ascending=False)
)

print("\nFlood counts by event:")
print(
    all_events_hazard.groupby("event")["flood_label"]
    .value_counts()
)

In [ ]:
features_to_check = [
    "elevation",
    "slope",
    "rainfall",
    "distance_to_coast",
    "population",
    "min_track_distance_km",
    "wind_distance_hazard"
]

print("Mean feature values by flood label:")
print(
    all_events_hazard
    .groupby("flood_label")[features_to_check]
    .mean()
    .T
)


In [ ]:
for event in ["Amphan", "Fani", "Yaas", "Remal"]:
    subset = all_events_hazard[
        all_events_hazard["event"] == event
    ]

    print(
        event,
        "| flood rate:",
        round(subset["flood_label"].mean(), 3),
        "| rainfall flood/non:",
        round(
            subset.loc[subset["flood_label"] == 1, "rainfall"].mean(), 2
        ),
        "/",
        round(
            subset.loc[subset["flood_label"] == 0, "rainfall"].mean(), 2
        )
    )

In [ ]:
for event in ["Amphan", "Fani", "Yaas", "Remal"]:
    subset = all_events_hazard[
        all_events_hazard["event"] == event
    ]

    print(
        event,
        "| elevation flood/non:",
        round(
            subset.loc[subset["flood_label"] == 1, "elevation"].mean(), 2
        ),
        "/",
        round(
            subset.loc[subset["flood_label"] == 0, "elevation"].mean(), 2
        ),
        "| coast distance flood/non:",
        round(
            subset.loc[subset["flood_label"] == 1, "distance_to_coast"].mean(), 2
        ),
        "/",
        round(
            subset.loc[subset["flood_label"] == 0, "distance_to_coast"].mean(), 2
        )
    )

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    model_hazard,
    X_train_h,
    y_train_h,
    cv=cv,
    scoring="roc_auc"
)

print("5-Fold Cross-Validation ROC-AUC:")
print(cv_scores)

print("\nMean ROC-AUC:", cv_scores.mean())
print("Std:", cv_scores.std())

In [ ]:
from sklearn.metrics import roc_auc_score

events_list = ["Amphan", "Fani", "Yaas", "Remal"]

for test_event in events_list:

    train_mask = events_hazard != test_event
    test_mask = events_hazard == test_event

    X_train = X_hazard[train_mask]
    y_train = y_hazard[train_mask]

    X_test = X_hazard[test_mask]
    y_test = y_hazard[test_mask]

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_prob)

    print(
        f"Test event: {test_event:7s} | "
        f"ROC-AUC: {auc:.3f}"
    )

In [ ]:
remal_track = tracks["Remal"].copy()

trajectory = remal_track[
    [
        "ISO_TIME",
        "LAT",
        "LON",
        "WMO_WIND",
        "WMO_PRES"
    ]
].copy()

trajectory.columns = [
    "time",
    "lat",
    "lon",
    "wind_kt",
    "pressure_hpa"
]

trajectory = trajectory.sort_values("time")

print("Remal trajectory points:", len(trajectory))
print(trajectory.head())
print("\nLast points:")
print(trajectory.tail())

In [ ]:
import json

trajectory_json = {
    "storm": "Remal",
    "track": [
        {
            "time": row["time"].isoformat(),
            "lat": float(row["lat"]),
            "lon": float(row["lon"]),
            "wind_kt": None if pd.isna(row["wind_kt"]) else float(row["wind_kt"]),
            "pressure_hpa": None if pd.isna(row["pressure_hpa"]) else float(row["pressure_hpa"])
        }
        for _, row in trajectory.iterrows()
    ]
}

print(json.dumps(trajectory_json, indent=2))

In [ ]:
with open("remal_trajectory.json", "w") as f:
    json.dump(trajectory_json, f, indent=2)

print("Saved: remal_trajectory.json")


In [ ]:
!pip -q install osmnx

In [ ]:
import osmnx as ox

study_polygon = ox.utils_geo.bbox_to_poly(
    (88.5, 21.5, 87.8, 22.5)
)

roads = ox.features_from_polygon(
    study_polygon,
    tags={"highway": True}
)

print("Road features:", len(roads))
print(roads[["highway"]].head())

In [ ]:
road_types = [
    "motorway",
    "trunk",
    "primary",
    "secondary",
    "tertiary",
    "unclassified",
    "residential"
]

road_segments = roads[
    roads["highway"].apply(
        lambda x: any(r in road_types for r in x)
        if isinstance(x, list)
        else x in road_types
    )
].copy()

print("Road segments:", len(road_segments))
print(
    road_segments["highway"]
    .explode()
    .value_counts()
)

In [ ]:
print(
    road_segments[
        ["highway", "name"]
    ].head(15)
)

In [ ]:
import geopandas as gpd
from shapely.geometry import LineString

track_coords = list(
    zip(
        trajectory["lon"],
        trajectory["lat"]
    )
)

remal_line = LineString(track_coords)

track_gdf = gpd.GeoDataFrame(
    {"storm": ["Remal"]},
    geometry=[remal_line],
    crs="EPSG:4326"
)

track_gdf

In [ ]:
roads_geo = road_segments[
    road_segments.geometry.notna()
].copy()

roads_geo = gpd.GeoDataFrame(
    roads_geo,
    geometry="geometry",
    crs="EPSG:4326"
)

# Project to metres for distance calculation
roads_proj = roads_geo.to_crs("EPSG:32645")
track_proj = track_gdf.to_crs("EPSG:32645")

corridor = track_proj.geometry.iloc[0].buffer(20_000)

exposed_roads = roads_proj[
    roads_proj.geometry.intersects(corridor)
].copy()

print("Exposed road segments:", len(exposed_roads))

In [ ]:
exposed_roads["length_km"] = (
    exposed_roads.geometry.length / 1000
)

print(
    "Total exposed road length:",
    round(exposed_roads["length_km"].sum(), 2),
    "km"
)

print("\nBy road type:")
print(
    exposed_roads["highway"]
    .explode()
    .value_counts()
)

In [ ]:
print("Remal track bounds:")
print(remal_line.bounds)

print("\nRoad bounds:")
print(roads_geo.total_bounds)

In [ ]:
from shapely.ops import nearest_points

road_union = roads_proj.geometry.union_all()

nearest_distance_m = track_proj.geometry.iloc[0].distance(road_union)

print(
    "Distance from Remal track to nearest road:",
    round(nearest_distance_m, 2),
    "metres"
)

In [ ]:
# Keep only Remal track points near our study area
study_track = trajectory[
    trajectory["lat"].between(21.5, 22.5) &
    trajectory["lon"].between(87.8, 88.5)
].copy()

print("Track points inside study area:", len(study_track))
print(study_track[["time", "lat", "lon", "wind_kt"]])

In [ ]:
# Build a corridor around the local portion of the track
track_coords = list(
    zip(study_track["lon"], study_track["lat"])
)

local_track_line = LineString(track_coords)

local_track_gdf = gpd.GeoDataFrame(
    {"storm": ["Remal"]},
    geometry=[local_track_line],
    crs="EPSG:4326"
)

local_track_proj = local_track_gdf.to_crs("EPSG:32645")

corridor = local_track_proj.geometry.iloc[0].buffer(20_000)

exposed_roads = roads_proj[
    roads_proj.geometry.intersects(corridor)
].copy()

print("Exposed road segments:", len(exposed_roads))

In [ ]:
power = ox.features_from_polygon(
    study_area_gdf.geometry.iloc[0],
    tags={"power": True}
)

print("Power features:", len(power))

print(
    power["power"]
    .value_counts()
)

In [ ]:
from shapely.geometry import box
import geopandas as gpd

study_area_geom = box(
    87.8, 21.5,
    88.5, 22.5
)

study_area_gdf = gpd.GeoDataFrame(
    {"name": ["study_area"]},
    geometry=[study_area_geom],
    crs="EPSG:4326"
)

print(study_area_gdf)

In [ ]:
power = ox.features_from_polygon(
    study_area_gdf.geometry.iloc[0],
    tags={"power": True}
)

print("Power features:", len(power))

print("\nPower feature types:")
print(power["power"].value_counts())

In [ ]:
print("Available columns:")
print(roads.columns.tolist())

print("\nPower column exists:", "power" in roads.columns)

if "power" in roads.columns:
    print("\nPower feature types:")
    print(roads["power"].value_counts(dropna=True).head(20))


In [ ]:
import osmnx as ox

ox.settings.use_cache = True
ox.settings.requests_timeout = 300
ox.settings.overpass_rate_limit = True

print("OSMnx configured.")

In [ ]:
substations = ox.features_from_polygon(
    study_area_gdf.geometry.iloc[0],
    tags={"power": "substation"}
)

print("Substations:", len(substations))
print(substations[["geometry", "power"]].head())

In [ ]:
# Clean road dataset
roads_geo = roads[
    roads.geometry.notna()
].copy()

roads_geo = gpd.GeoDataFrame(
    roads_geo,
    geometry="geometry",
    crs="EPSG:4326"
)

# Project to metres
roads_proj = roads_geo.to_crs("EPSG:32645")

roads_proj["length_km"] = (
    roads_proj.geometry.length / 1000
)

print("Total road segments:", len(roads_proj))
print(
    "Total road length:",
    round(roads_proj["length_km"].sum(), 2),
    "km"
)

print("\nRoad types:")
print(
    roads_proj["highway"]
    .explode()
    .value_counts()
)

In [ ]:
road_summary = {
    "total_segments": int(len(roads_proj)),
    "total_length_km": round(
        roads_proj["length_km"].sum(), 2
    ),
    "road_types": (
        roads_proj["highway"]
        .explode()
        .value_counts()
        .to_dict()
    )
}

road_summary

In [ ]:
# Use the Remal ML samples
remal_samples = remal.copy()

# Convert sample coordinates into GeoDataFrame
remal_points = gpd.GeoDataFrame(
    remal_samples,
    geometry=gpd.points_from_xy(
        remal_samples["lon"],
        remal_samples["lat"]
    ),
    crs="EPSG:4326"
)

# Project to metres
remal_points_proj = remal_points.to_crs("EPSG:32645")

print("Remal sample points:", len(remal_points_proj))
print("Flood samples:", int(remal_points_proj["flood_label"].sum()))

In [ ]:
# Create a 100 m influence zone around each flood sample
flood_points = remal_points_proj[
    remal_points_proj["flood_label"] == 1
].copy()

flood_zone = flood_points.geometry.buffer(100).union_all()

affected_roads = roads_proj[
    roads_proj.geometry.intersects(flood_zone)
].copy()

print("Road segments intersecting flood-sample zones:",
      len(affected_roads))

print(
    "Affected road length:",
    round(affected_roads["length_km"].sum(), 2),
    "km"
)

In [ ]:
infrastructure_exposure = {
    "storm": "Remal",
    "flood_sample_points": int(len(flood_points)),
    "total_sample_points": int(len(remal_points_proj)),
    "potentially_exposed_road_segments": int(len(affected_roads)),
    "potentially_exposed_road_length_km": round(
        affected_roads["length_km"].sum(), 2
    )
}

infrastructure_exposure

In [ ]:
affected_road_types = (
    affected_roads["highway"]
    .explode()
    .value_counts()
    .to_dict()
)

infrastructure_exposure["affected_road_types"] = {
    str(k): int(v)
    for k, v in affected_road_types.items()
}

infrastructure_exposure

In [ ]:
gemini_input = {
    "storm": "Remal",

    "trajectory": trajectory_json,

    "hazard": {
        "flood_sample_points": int(len(flood_points)),
        "total_sample_points": int(len(remal_points_proj)),
        "potentially_flood_affected_fraction": round(
            len(flood_points) / len(remal_points_proj),
            3
        )
    },

    "infrastructure": infrastructure_exposure
}

print(json.dumps(gemini_input, indent=2, default=str))

In [ ]:
import json

gemini_text = json.dumps(
    gemini_input,
    indent=2,
    default=str
)

print("Characters:", len(gemini_text))
print("Approx. words:", len(gemini_text.split()))

In [ ]:
!pip -q install -U google-genai

from google import genai
import os

In [ ]:
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

print("API key loaded:", api_key is not None)

In [ ]:
from google import genai

client = genai.Client(api_key=api_key)

print("Gemini client initialized.")

In [ ]:
models = client.models.list()

for model in models:
    print(model.name)

In [ ]:
response = client.models.generate_content(
    model="gemini-flash-latest",
    contents="In one sentence, explain what a cyclone infrastructure vulnerability forecaster does."
)

print(response.text)

In [ ]:
prompt = f"""
You are the disaster-risk intelligence layer of a cyclone
impact and infrastructure vulnerability forecasting system.

Analyze the structured data below.

IMPORTANT RULES:
- The Remal trajectory is an OBSERVED historical track, not a forecast.
- Flood values are prototype Sentinel-1 change-detection estimates.
- Road exposure is a potential exposure estimate, NOT proof that roads were flooded.
- Do not invent missing data.
- Clearly distinguish observations from interpretations.
- Base your analysis only on the supplied data.

Return the following sections:

1. HAZARD SUMMARY
Summarize the cyclone trajectory and available hazard indicators.

2. INFRASTRUCTURE EXPOSURE
Interpret the potentially exposed road network and flood-sample information.

3. KEY RISK FACTORS
Identify the most relevant risk factors visible in the supplied data.

4. PRIORITY AREAS
Identify what types of locations/infrastructure should receive attention based on the supplied evidence.

5. PREPAREDNESS ACTIONS
Give practical disaster-preparedness actions that could follow from these findings.

6. DATA LIMITATIONS
Clearly state the major limitations and uncertainties in this prototype.

DATA:
{gemini_text}
"""

response = client.models.generate_content(
    model="gemini-3.7-flash",
    contents=prompt
)

print(response.text)

In [ ]:
response = client.models.generate_content(
    model="gemini-3.7-flash",
    contents="Say OK."
)

print(response.text)

In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Say OK."
)

print(response.text)

In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)

In [ ]:
prompt = f"""
You are the disaster-risk intelligence layer of a cyclone
impact and infrastructure vulnerability forecasting system.

Analyze ONLY the structured data provided below.

IMPORTANT RULES:
- The Remal trajectory is an OBSERVED historical track, not a forecast.
- Flood values are prototype Sentinel-1 change-detection estimates.
- Road exposure is a potential exposure estimate, NOT proof that roads were flooded.
- Do not invent missing data.
- Do not claim causation or correlation unless explicitly supported
  by the supplied data.
- Distinguish observed measurements from interpretations and recommendations.
- Do not infer that a location was affected simply because it is near
  the cyclone track.
- Base every statement on the supplied data.
- If information is unavailable, explicitly say "Not available from supplied data".

Return ONLY valid JSON.
Do not use Markdown.
Do not add explanations outside the JSON.

Use EXACTLY this structure:

{{
  "hazard_summary": {{
    "trajectory": "...",
    "intensity_indicators": "...",
    "observed_vs_interpreted": "..."
  }},

  "infrastructure_exposure": {{
    "road_exposure": "...",
    "flood_information": "...",
    "infrastructure_interpretation": "..."
  }},

  "key_risk_factors": [
    {{
      "factor": "...",
      "evidence": "...",
      "confidence": "high/medium/low"
    }}
  ],

  "priority_areas": [
    {{
      "area_or_category": "...",
      "reason": "...",
      "evidence": "...",
      "priority": "high/medium/low"
    }}
  ],

  "preparedness_actions": [
    {{
      "action": "...",
      "reason": "..."
    }}
  ],

  "data_limitations": [
    "..."
  ]
}}

DATA:
{gemini_text}
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)

In [ ]:
print([name for name in globals() if not name.startswith("_")])

In [ ]:
import os

for file in os.listdir("/content"):
    print(file)

.config
Fani_ML_Dataset.csv
Remal_ML_Dataset.csv
ibtracs.NI.list.v04r01.csv
Yaas_ML_Dataset.csv
Cyclone_Geospatial_Features.csv
sample_data


In [ ]:
import pandas as pd

fani_df = pd.read_csv("/content/Fani_ML_Dataset.csv")
remal_df = pd.read_csv("/content/Remal_ML_Dataset.csv")
yaas_df = pd.read_csv("/content/Yaas_ML_Dataset.csv")
geo_df = pd.read_csv("/content/Cyclone_Geospatial_Features.csv")

print("Fani:", fani_df.shape)
print("Remal:", remal_df.shape)
print("Yaas:", yaas_df.shape)
print("Geospatial:", geo_df.shape)

Fani: (994, 8)
Remal: (994, 8)
Yaas: (994, 8)
Geospatial: (2560, 7)


In [ ]:
print("Fani columns:")
print(fani_df.columns.tolist())

print("\nRemal columns:")
print(remal_df.columns.tolist())

print("\nYaas columns:")
print(yaas_df.columns.tolist())

print("\nGeospatial columns:")
print(geo_df.columns.tolist())

Fani columns:
['system:index', 'distance_to_coast', 'elevation', 'flood_label', 'population', 'rainfall', 'slope', '.geo']

Remal columns:
['system:index', 'distance_to_coast', 'elevation', 'flood_label', 'population', 'rainfall', 'slope', '.geo']

Yaas columns:
['system:index', 'distance_to_coast', 'elevation', 'flood_label', 'population', 'rainfall', 'slope', '.geo']

Geospatial columns:
['system:index', 'distance_to_coast', 'elevation', 'population', 'rainfall', 'slope', '.geo']


In [ ]:
# Create a compact text representation of the available cyclone data

def dataset_summary(name, df):
    text = f"\n===== {name} =====\n"
    text += f"Rows: {len(df)}\n"
    text += f"Columns: {', '.join(df.columns.astype(str))}\n\n"

    # Include representative records
    text += "Sample records:\n"
    text += df.head(10).to_string(index=False)

    # Include numerical summary where available
    numeric_cols = df.select_dtypes(include="number").columns

    if len(numeric_cols) > 0:
        text += "\n\nNumerical summary:\n"
        text += df[numeric_cols].describe().round(3).to_string()

    return text


gemini_text = (
    dataset_summary("FANI CYCLONE DATA", fani_df)
    + dataset_summary("REMAL CYCLONE DATA", remal_df)
    + dataset_summary("YAAS CYCLONE DATA", yaas_df)
    + dataset_summary("COMBINED GEOSPATIAL FEATURES", geo_df)
)

print(gemini_text[:15000])


===== FANI CYCLONE DATA =====
Rows: 994
Columns: system:index, distance_to_coast, elevation, flood_label, population, rainfall, slope, .geo

Sample records:
 system:index  distance_to_coast  elevation  flood_label  population  rainfall    slope                                                                  .geo
            0          45.650849   2.288315            0    5.271847 90.346390 0.017231   {"type":"Point","coordinates":[87.8178098886383,21.78429354153335]}
            1          23.706539   3.032541            0   10.967005 52.267288 0.004937 {"type":"Point","coordinates":[88.22603996192875,22.113556873477222]}
            3         110.724884   2.411536            1   11.175486 78.039158 0.017231 {"type":"Point","coordinates":[87.86348854699396,21.926739367524743]}
            6           9.219544   2.475800            0   25.727804 47.103163 0.004937 {"type":"Point","coordinates":[88.25472781239424,22.184625025267362]}
            8          69.079664   5.907362         

In [ ]:
print("\nLength of Gemini input:", len(gemini_text))


Length of Gemini input: 10826


In [ ]:
import json

prompt = f"""
You are the disaster-risk intelligence layer of a cyclone
impact and infrastructure vulnerability forecasting system.

Analyze ONLY the structured data provided below.

IMPORTANT RULES:
- The Remal trajectory is an OBSERVED historical track, not a forecast.
- Flood values are prototype Sentinel-1 change-detection estimates.
- Road exposure is a potential exposure estimate, NOT proof that roads were flooded.
- Do not invent missing data.
- Do not claim causation or correlation unless explicitly supported
  by the supplied data.
- Distinguish observed measurements from interpretations and recommendations.
- Do not infer that a location was affected simply because it is near
  the cyclone track.
- Base every statement on the supplied data.
- If information is unavailable, explicitly say:
  "Not available from supplied data."

Return ONLY valid JSON.
Do not use Markdown.
Do not add explanations outside the JSON.

Use EXACTLY this structure:

{{
  "hazard_summary": {{
    "trajectory": "...",
    "intensity_indicators": "...",
    "observed_vs_interpreted": "..."
  }},

  "infrastructure_exposure": {{
    "road_exposure": "...",
    "flood_information": "...",
    "infrastructure_interpretation": "..."
  }},

  "key_risk_factors": [
    {{
      "factor": "...",
      "evidence": "...",
      "confidence": "high/medium/low"
    }}
  ],

  "priority_areas": [
    {{
      "area_or_category": "...",
      "reason": "...",
      "evidence": "...",
      "priority": "high/medium/low"
    }}
  ],

  "preparedness_actions": [
    {{
      "action": "...",
      "reason": "..."
    }}
  ],

  "data_limitations": [
    "..."
  ]
}}

DATA:
{gemini_text}
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)

NameError: name 'client' is not defined

In [ ]:
!pip -q install -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 10.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.


In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [ ]:
prompt = f"""
You are the disaster-risk intelligence layer of a cyclone
impact and infrastructure vulnerability forecasting system.

Analyze ONLY the structured data provided below.

IMPORTANT RULES:
- The Remal trajectory is an OBSERVED historical track, not a forecast.
- Flood values are prototype Sentinel-1 change-detection estimates.
- Road exposure is a potential exposure estimate, NOT proof that roads were flooded.
- Do not invent missing data.
- Do not claim causation or correlation unless explicitly supported
  by the supplied data.
- Distinguish observed measurements from interpretations and recommendations.
- Do not infer that a location was affected simply because it is near
  the cyclone track.
- Base every statement on the supplied data.
- If information is unavailable, explicitly say:
  "Not available from supplied data."

Return ONLY valid JSON.
Do not use Markdown.
Do not add explanations outside the JSON.

Use EXACTLY this structure:

{{
  "hazard_summary": {{
    "trajectory": "...",
    "intensity_indicators": "...",
    "observed_vs_interpreted": "..."
  }},

  "infrastructure_exposure": {{
    "road_exposure": "...",
    "flood_information": "...",
    "infrastructure_interpretation": "..."
  }},

  "key_risk_factors": [
    {{
      "factor": "...",
      "evidence": "...",
      "confidence": "high/medium/low"
    }}
  ],

  "priority_areas": [
    {{
      "area_or_category": "...",
      "reason": "...",
      "evidence": "...",
      "priority": "high/medium/low"
    }}
  ],

  "preparedness_actions": [
    {{
      "action": "...",
      "reason": "..."
    }}
  ],

  "data_limitations": [
    "..."
  ]
}}

DATA:
{gemini_text}
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)

{
  "hazard_summary": {
    "trajectory": "The Remal trajectory is an observed historical track, not a forecast.",
    "intensity_indicators": "For Remal, rainfall ranged from a min of 87.336 to a max of 168.132 with a mean of 126.504. Elevation ranged from 0.500 to 12.909 with a mean of 4.514. Distance to coast ranged from 0.000 to 249.419 with a mean of 53.488.",
    "observed_vs_interpreted": "Flood values are prototype Sentinel-1 change-detection estimates. Road exposure is a potential exposure estimate, NOT proof that roads were flooded."
  },
  "infrastructure_exposure": {
    "road_exposure": "Not available from supplied data.",
    "flood_information": "For Remal, flood_label count is 994 with a mean of 0.169, std of 0.375, min of 0.000, 25% of 0.000, 50% of 0.000, 75% of 0.000, and max of 1.000.",
    "infrastructure_interpretation": "Not available from supplied data."
  },
  "key_risk_factors": [
    {
      "factor": "Rainfall",
      "evidence": "Remal rainfall mean is 126.

In [ ]:
import os

print("Files currently in Colab:\n")

for file in os.listdir("/content"):
    print(file)

Files currently in Colab:

.config
Fani_ML_Dataset.csv
Remal_ML_Dataset.csv
ibtracs.NI.list.v04r01.csv
Yaas_ML_Dataset.csv
Cyclone_Geospatial_Features.csv
sample_data


In [ ]:
import pandas as pd

geo_df = pd.read_csv("/content/Cyclone_Geospatial_Features.csv")

print(geo_df.columns.tolist())
print(geo_df.head())

['system:index', 'distance_to_coast', 'elevation', 'population', 'rainfall', 'slope', '.geo']
   system:index  distance_to_coast  elevation  population    rainfall  \
0             0          14.560220   2.342535    5.211901  387.064301   
1             1           7.211103   2.873379   10.967005  321.451001   
2             3          28.284271   2.392084   10.974935  374.830141   
3             6          29.068884   3.173908   26.200888  300.842326   
4             8          46.615448   4.050722   12.444121  285.933142   

      slope                                               .geo  
0  0.017231  {"type":"Point","coordinates":[87.817809888638...  
1  0.004937  {"type":"Point","coordinates":[88.226039961928...  
2  0.017231  {"type":"Point","coordinates":[87.863488546993...  
3  0.004937  {"type":"Point","coordinates":[88.254727812394...  
4  0.004937  {"type":"Point","coordinates":[88.453711387282...  


In [ ]:
# Build a clean, compact input for Gemini

gemini_text = f"""
CYCLONE ML DATA
================

FANI
Rows: {len(fani_df)}
Flood-positive rate: {fani_df['flood_label'].mean():.3f}
Rainfall mean: {fani_df['rainfall'].mean():.3f}
Rainfall min: {fani_df['rainfall'].min():.3f}
Rainfall max: {fani_df['rainfall'].max():.3f}
Elevation mean: {fani_df['elevation'].mean():.3f}
Elevation min: {fani_df['elevation'].min():.3f}
Elevation max: {fani_df['elevation'].max():.3f}
Distance-to-coast mean: {fani_df['distance_to_coast'].mean():.3f}
Population mean: {fani_df['population'].mean():.3f}
Slope mean: {fani_df['slope'].mean():.3f}

REMAL
Rows: {len(remal_df)}
Flood-positive rate: {remal_df['flood_label'].mean():.3f}
Rainfall mean: {remal_df['rainfall'].mean():.3f}
Rainfall min: {remal_df['rainfall'].min():.3f}
Rainfall max: {remal_df['rainfall'].max():.3f}
Elevation mean: {remal_df['elevation'].mean():.3f}
Elevation min: {remal_df['elevation'].min():.3f}
Elevation max: {remal_df['elevation'].max():.3f}
Distance-to-coast mean: {remal_df['distance_to_coast'].mean():.3f}
Population mean: {remal_df['population'].mean():.3f}
Slope mean: {remal_df['slope'].mean():.3f}

YAAS
Rows: {len(yaas_df)}
Flood-positive rate: {yaas_df['flood_label'].mean():.3f}
Rainfall mean: {yaas_df['rainfall'].mean():.3f}
Rainfall min: {yaas_df['rainfall'].min():.3f}
Rainfall max: {yaas_df['rainfall'].max():.3f}
Elevation mean: {yaas_df['elevation'].mean():.3f}
Elevation min: {yaas_df['elevation'].min():.3f}
Elevation max: {yaas_df['elevation'].max():.3f}
Distance-to-coast mean: {yaas_df['distance_to_coast'].mean():.3f}
Population mean: {yaas_df['population'].mean():.3f}
Slope mean: {yaas_df['slope'].mean():.3f}

COMBINED GEOSPATIAL DATA
========================
Rows: {len(geo_df)}
Rainfall mean: {geo_df['rainfall'].mean():.3f}
Rainfall min: {geo_df['rainfall'].min():.3f}
Rainfall max: {geo_df['rainfall'].max():.3f}
Elevation mean: {geo_df['elevation'].mean():.3f}
Elevation min: {geo_df['elevation'].min():.3f}
Elevation max: {geo_df['elevation'].max():.3f}
Distance-to-coast mean: {geo_df['distance_to_coast'].mean():.3f}
Population mean: {geo_df['population'].mean():.3f}
Slope mean: {geo_df['slope'].mean():.3f}

AVAILABLE DATA:
- Environmental/geospatial features: rainfall, elevation, distance to coast,
  population and slope.
- Flood labels are available for the historical cyclone datasets.
- Cyclone trajectory visualization is handled separately by the frontend.
- No road-network exposure dataset is included in this notebook input.
- No infrastructure-specific dataset is included in this notebook input.
"""

print(gemini_text)


CYCLONE ML DATA

FANI
Rows: 994
Flood-positive rate: 0.154
Rainfall mean: 63.598
Rainfall min: 47.103
Rainfall max: 95.695
Elevation mean: 4.514
Elevation min: 0.500
Elevation max: 12.909
Distance-to-coast mean: 53.488
Population mean: 19.011
Slope mean: 0.012

REMAL
Rows: 994
Flood-positive rate: 0.169
Rainfall mean: 126.504
Rainfall min: 87.336
Rainfall max: 168.132
Elevation mean: 4.514
Elevation min: 0.500
Elevation max: 12.909
Distance-to-coast mean: 53.488
Population mean: 19.011
Slope mean: 0.012

YAAS
Rows: 994
Flood-positive rate: 0.252
Rainfall mean: 101.643
Rainfall min: 23.751
Rainfall max: 162.238
Elevation mean: 4.514
Elevation min: 0.500
Elevation max: 12.909
Distance-to-coast mean: 53.488
Population mean: 19.011
Slope mean: 0.012

COMBINED GEOSPATIAL DATA
Rows: 2560
Rainfall mean: 311.836
Rainfall min: 228.806
Rainfall max: 390.797
Elevation mean: 4.539
Elevation min: 0.500
Elevation max: 12.788
Distance-to-coast mean: 14.945
Population mean: 20.181
Slope mean: 0.012



In [20]:
import pandas as pd
import glob
import os

files = glob.glob('/content/*')

print("Uploaded files:")
for f in files:
    print(os.path.basename(f))

Uploaded files:
Fani_ML_Dataset.csv
Remal_ML_Dataset.csv
ibtracs.NI.list.v04r01.csv
Yaas_ML_Dataset.csv
cyclone_power_infrastructure.csv
cyclone_roads.csv
Cyclone_Geospatial_Features.csv
sample_data


In [21]:
road_file = [f for f in files if 'road' in os.path.basename(f).lower()][0]
power_file = [f for f in files if 'power' in os.path.basename(f).lower()][0]

roads_df = pd.read_csv(road_file)
power_df = pd.read_csv(power_file)

print("Road dataset:", roads_df.shape)
print("Power dataset:", power_df.shape)

print("\nRoad columns:")
print(roads_df.columns.tolist())

print("\nPower columns:")
print(power_df.columns.tolist())

Road dataset: (87306, 12)
Power dataset: (39168, 25)

Road columns:
['system:index', 'GP_RAV', 'GP_RCY', 'GP_REX', 'GP_RRG', 'GP_RSE', 'GP_RSI', 'GP_RSY', 'GP_RTP', 'Shape_Leng', 'gp_gripreg', '.geo']

Power columns:
['system:index', 'cap_year', 'capacitymw', 'comm_year', 'country', 'country_lg', 'fuel1', 'fuel2', 'fuel3', 'fuel4', 'gppd_idnr', 'gwh_2013', 'gwh_2014', 'gwh_2015', 'gwh_2016', 'gwh_estimt', 'infrastructure_type', 'latitude', 'longitude', 'name', 'owner', 'source', 'src_latlon', 'url', '.geo']


/tmp/ipykernel_513/1005812177.py:5: DtypeWarning: Columns (4,5,6,7,10,19,20,22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  power_df = pd.read_csv(power_file)


In [22]:
print("========== ROADS ==========")
display(roads_df.head(3))

print("\n========== POWER INFRASTRUCTURE ==========")
display(power_df.head(3))

print("\nPower infrastructure types:")
if 'infrastructure_type' in power_df.columns:
    print(power_df['infrastructure_type'].value_counts())

========== ROADS ==========


,system:index,GP_RAV,GP_RCY,GP_REX,GP_RRG,GP_RSE,GP_RSI,GP_RSY,GP_RTP,Shape_Leng,gp_gripreg,.geo
0,000700000000000033c8,1,50,1,18,0,51,2006,4,0.000216,6,"{""type"":""LineString"",""coordinates"":[[90.498459..."
1,000700000000000033c9,1,50,1,18,0,51,2006,4,0.000020,6,"{""type"":""LineString"",""coordinates"":[[90.498668..."
2,000700000000000033ca,1,50,1,18,0,51,2006,4,0.000040,6,"{""type"":""LineString"",""coordinates"":[[90.498708..."



========== POWER INFRASTRUCTURE ==========


,system:index,cap_year,capacitymw,comm_year,country,country_lg,fuel1,fuel2,fuel3,fuel4,...,gwh_estimt,infrastructure_type,latitude,longitude,name,owner,source,src_latlon,url,.geo
0,1_00000000000000005ecf,0.0,102.0,2015.0,MMR,Myanmar,Gas,NaN,NaN,NaN,...,484.911644,power_plant,21.6000,86.2130,Kyaukse,Myanmar Electric Power Enterprise (MEPE),Power Mag,WRI,http://www.powermag.com/kyaukse-power-plant-ky...,"{""type"":""Point"",""coordinates"":[86.212988167123..."
1,1_00000000000000005ea1,0.0,1000.0,2012.0,IND,India,Coal,Oil,NaN,NaN,...,0.000000,power_plant,24.3858,85.5517,KODERMA,NaN,Central Electricity Authority,WRI,http://www.cea.nic.in/,"{""type"":""Point"",""coordinates"":[85.551677789372..."
2,1_00000000000000005ea2,0.0,890.0,1990.0,IND,India,Coal,Oil,NaN,NaN,...,0.000000,power_plant,23.7376,86.1273,CHANDRAPURA,NaN,Central Electricity Authority,WRI,http://www.cea.nic.in/,"{""type"":""Point"",""coordinates"":[86.127305968730..."



Power infrastructure types:
infrastructure_type
power_grid     39112
power_plant       56
Name: count, dtype: int64


In [23]:
print("ROAD GEOMETRY SAMPLE")
print(roads_df['.geo'].iloc[0])

print("\nPOWER GEOMETRY SAMPLE")
print(power_df['.geo'].iloc[0])

ROAD GEOMETRY SAMPLE
{"type":"LineString","coordinates":[[90.49845917742113,22.235838036033233],[90.49853944124008,22.2358603315385],[90.49866875517064,22.235891545245877]]}

POWER GEOMETRY SAMPLE
{"type":"Point","coordinates":[86.21298816712361,21.599992987106535]}


In [24]:
geo_df = pd.read_csv('/content/Cyclone_Geospatial_Features.csv')

print("Geospatial dataset:", geo_df.shape)
print("\nColumns:")
print(geo_df.columns.tolist())

display(geo_df.head(3))

Geospatial dataset: (2560, 7)

Columns:
['system:index', 'distance_to_coast', 'elevation', 'population', 'rainfall', 'slope', '.geo']


,system:index,distance_to_coast,elevation,population,rainfall,slope,.geo
0,0,14.560220,2.342535,5.211901,387.064301,0.017231,"{""type"":""Point"",""coordinates"":[87.817809888638..."
1,1,7.211103,2.873379,10.967005,321.451001,0.004937,"{""type"":""Point"",""coordinates"":[88.226039961928..."
2,3,28.284271,2.392084,10.974935,374.830141,0.017231,"{""type"":""Point"",""coordinates"":[87.863488546993..."


In [25]:
print("Columns containing risk/prediction/flood:")
for col in geo_df.columns:
    if any(x in col.lower() for x in ['risk', 'predict', 'flood', 'prob']):
        print(col)

print("\nDataset info:")
print(geo_df.shape)
print(geo_df.dtypes)

Columns containing risk/prediction/flood:

Dataset info:
(2560, 7)
system:index           int64
distance_to_coast    float64
elevation            float64
population           float64
rainfall             float64
slope                float64
.geo                  object
dtype: object


In [26]:
import json
import numpy as np
import pandas as pd

def extract_point_coordinates(geo):
    data = json.loads(geo)
    coords = data['coordinates']
    return pd.Series({
        'longitude': coords[0],
        'latitude': coords[1]
    })

coordinates = geo_df['.geo'].apply(extract_point_coordinates)

geo_df = pd.concat(
    [geo_df.drop(columns=['.geo']), coordinates],
    axis=1
)

print("Geospatial dataset:", geo_df.shape)
display(geo_df.head(3))

Geospatial dataset: (2560, 8)


,system:index,distance_to_coast,elevation,population,rainfall,slope,longitude,latitude
0,0,14.560220,2.342535,5.211901,387.064301,0.017231,87.817810,21.784294
1,1,7.211103,2.873379,10.967005,321.451001,0.004937,88.226040,22.113557
2,3,28.284271,2.392084,10.974935,374.830141,0.017231,87.863489,21.926739


In [27]:
fani_df = pd.read_csv('/content/Fani_ML_Dataset.csv')
remal_df = pd.read_csv('/content/Remal_ML_Dataset.csv')
yaas_df = pd.read_csv('/content/Yaas_ML_Dataset.csv')

ml_df = pd.concat(
    [fani_df, remal_df, yaas_df],
    ignore_index=True
)

print("Combined ML dataset:", ml_df.shape)
print("Columns:")
print(ml_df.columns.tolist())

print("\nFlood label distribution:")
print(ml_df['flood_label'].value_counts())

Combined ML dataset: (2982, 8)
Columns:
['system:index', 'distance_to_coast', 'elevation', 'flood_label', 'population', 'rainfall', 'slope', '.geo']

Flood label distribution:
flood_label
0    2411
1     571
Name: count, dtype: int64


In [28]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

feature_columns = [
    'distance_to_coast',
    'elevation',
    'population',
    'rainfall',
    'slope'
]

X = ml_df[feature_columns]
y = ml_df['flood_label']

X_geo = geo_df[feature_columns]

print("Training features:", X.shape)
print("Prediction features:", X_geo.shape)

print("\nMissing training values:")
print(X.isnull().sum())

print("\nMissing prediction values:")
print(X_geo.isnull().sum())

Training features: (2982, 5)
Prediction features: (2560, 5)

Missing training values:
distance_to_coast    0
elevation            0
population           0
rainfall             0
slope                0
dtype: int64

Missing prediction values:
distance_to_coast    0
elevation            0
population           0
rainfall             0
slope                0
dtype: int64


In [29]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_model.fit(X, y)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [30]:
geo_df['flood_probability'] = rf_model.predict_proba(X_geo)[:, 1]
geo_df['flood_prediction'] = rf_model.predict(X_geo)

geo_df['risk_level'] = pd.cut(
    geo_df['flood_probability'],
    bins=[-0.01, 0.30, 0.60, 1.01],
    labels=['Low', 'Moderate', 'High']
)

print("Flood-risk predictions generated.")
print("\nRisk distribution:")
print(geo_df['risk_level'].value_counts())

display(
    geo_df[
        ['latitude', 'longitude', 'flood_probability',
         'flood_prediction', 'risk_level']
    ].head(10)
)

Flood-risk predictions generated.

Risk distribution:
risk_level
Low         2443
Moderate     117
High           0
Name: count, dtype: int64


,latitude,longitude,flood_probability,flood_prediction,risk_level
0,21.784294,87.817810,0.243333,0,Low
1,22.113557,88.226040,0.116667,0,Low
2,21.926739,87.863489,0.116667,0,Low
3,22.184625,88.254728,0.196667,0,Low
4,22.153667,88.453711,0.150000,0,Low
5,22.123321,88.229020,0.146667,0,Low
6,22.097959,88.111126,0.150000,0,Low
7,22.072864,88.277190,0.296667,0,Low
8,21.930520,87.942572,0.193333,0,Low
9,21.891835,87.933819,0.113333,0,Low


In [31]:
risk_summary = {
    'total_locations': len(geo_df),
    'high_risk_locations': int((geo_df['risk_level'] == 'High').sum()),
    'moderate_risk_locations': int((geo_df['risk_level'] == 'Moderate').sum()),
    'low_risk_locations': int((geo_df['risk_level'] == 'Low').sum()),
    'mean_flood_probability': float(geo_df['flood_probability'].mean()),
    'max_flood_probability': float(geo_df['flood_probability'].max())
}

print("========== FLOOD RISK SUMMARY ==========")
for key, value in risk_summary.items():
    print(f"{key}: {value}")

========== FLOOD RISK SUMMARY ==========
total_locations: 2560
high_risk_locations: 0
moderate_risk_locations: 117
low_risk_locations: 2443
mean_flood_probability: 0.16351171875
max_flood_probability: 0.5266666666666666


In [32]:
import geopandas as gpd
from shapely.geometry import Point, shape
import json

# Risk locations
risk_gdf = gpd.GeoDataFrame(
    geo_df,
    geometry=[
        Point(lon, lat)
        for lon, lat in zip(
            geo_df['longitude'],
            geo_df['latitude']
        )
    ],
    crs='EPSG:4326'
)

# Roads
roads_gdf = gpd.GeoDataFrame(
    roads_df,
    geometry=roads_df['.geo'].apply(
        lambda x: shape(json.loads(x))
    ),
    crs='EPSG:4326'
)

# Power infrastructure
power_gdf = gpd.GeoDataFrame(
    power_df,
    geometry=power_df['.geo'].apply(
        lambda x: shape(json.loads(x))
    ),
    crs='EPSG:4326'
)

print("Risk points:", len(risk_gdf))
print("Roads:", len(roads_gdf))
print("Power infrastructure:", len(power_gdf))

Risk points: 2560
Roads: 87306
Power infrastructure: 39168


In [33]:
high_risk = risk_gdf[
    risk_gdf['risk_level'] == 'High'
].copy()

moderate_risk = risk_gdf[
    risk_gdf['risk_level'] == 'Moderate'
].copy()

print("High-risk locations:", len(high_risk))
print("Moderate-risk locations:", len(moderate_risk))
print("Low-risk locations:", len(risk_gdf) - len(high_risk) - len(moderate_risk))

High-risk locations: 0
Moderate-risk locations: 117
Low-risk locations: 2443


In [34]:
# Use a small geographic radius around each predicted risk point.
# 0.05 degrees is roughly a few kilometres in this region.

high_risk_buffer = high_risk.copy()
high_risk_buffer['geometry'] = high_risk_buffer.geometry.buffer(0.05)

roads_at_risk = gpd.sjoin(
    roads_gdf,
    high_risk_buffer[['geometry']],
    predicate='intersects',
    how='inner'
)

power_at_risk = gpd.sjoin(
    power_gdf,
    high_risk_buffer[['geometry']],
    predicate='intersects',
    how='inner'
)

print("========== INFRASTRUCTURE AT HIGH RISK ==========")
print("Road features near high-risk locations:", len(roads_at_risk))
print("Power infrastructure near high-risk locations:", len(power_at_risk))

if 'infrastructure_type' in power_at_risk.columns:
    print("\nPower infrastructure breakdown:")
    print(power_at_risk['infrastructure_type'].value_counts())

========== INFRASTRUCTURE AT HIGH RISK ==========
Road features near high-risk locations: 0
Power infrastructure near high-risk locations: 0

Power infrastructure breakdown:
Series([], Name: count, dtype: int64)


/tmp/ipykernel_513/513343619.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  high_risk_buffer['geometry'] = high_risk_buffer.geometry.buffer(0.05)


In [35]:
# Each infrastructure feature may intersect multiple high-risk points,
# so count unique infrastructure features rather than spatial matches.

roads_at_risk_unique = roads_at_risk.drop_duplicates(
    subset=['system:index']
)

power_at_risk_unique = power_at_risk.drop_duplicates(
    subset=['system:index']
)

print("Unique road features at high risk:",
      len(roads_at_risk_unique))

print("Unique power infrastructure features at high risk:",
      len(power_at_risk_unique))

Unique road features at high risk: 0
Unique power infrastructure features at high risk: 0


In [36]:
road_exposure_pct = (
    len(roads_at_risk_unique) / len(roads_gdf)
) * 100

power_exposure_pct = (
    len(power_at_risk_unique) / len(power_gdf)
) * 100

infrastructure_risk_summary = {
    'total_roads': len(roads_gdf),
    'roads_near_high_risk': len(roads_at_risk_unique),
    'road_exposure_percent': round(road_exposure_pct, 2),

    'total_power_infrastructure': len(power_gdf),
    'power_near_high_risk': len(power_at_risk_unique),
    'power_exposure_percent': round(power_exposure_pct, 2)
}

print("========== INFRASTRUCTURE RISK SUMMARY ==========")
for key, value in infrastructure_risk_summary.items():
    print(f"{key}: {value}")

========== INFRASTRUCTURE RISK SUMMARY ==========
total_roads: 87306
roads_near_high_risk: 0
road_exposure_percent: 0.0
total_power_infrastructure: 39168
power_near_high_risk: 0
power_exposure_percent: 0.0


In [37]:
gemini_project_summary = {
    'risk': risk_summary,
    'infrastructure': infrastructure_risk_summary
}

if 'infrastructure_type' in power_at_risk_unique.columns:
    gemini_project_summary['power_breakdown'] = (
        power_at_risk_unique['infrastructure_type']
        .value_counts()
        .to_dict()
    )

print("========== GEMINI PROJECT SUMMARY ==========")
print(gemini_project_summary)

========== GEMINI PROJECT SUMMARY ==========
{'risk': {'total_locations': 2560, 'high_risk_locations': 0, 'moderate_risk_locations': 117, 'low_risk_locations': 2443, 'mean_flood_probability': 0.16351171875, 'max_flood_probability': 0.5266666666666666}, 'infrastructure': {'total_roads': 87306, 'roads_near_high_risk': 0, 'road_exposure_percent': 0.0, 'total_power_infrastructure': 39168, 'power_near_high_risk': 0, 'power_exposure_percent': 0.0}, 'power_breakdown': {}}


In [38]:
print("Flood probability statistics:")
print(
    geo_df['flood_probability'].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

Flood probability statistics:
count    2560.000000
mean        0.163512
std         0.070715
min         0.026667
25%         0.113333
50%         0.150000
75%         0.200000
90%         0.256667
95%         0.296667
max         0.526667
Name: flood_probability, dtype: float64


In [39]:
geo_df['risk_level'] = pd.cut(
    geo_df['flood_probability'],
    bins=[-0.01, 0.20, 0.35, 1.01],
    labels=['Low', 'Moderate', 'High']
)

print("========== UPDATED RISK DISTRIBUTION ==========")
print(geo_df['risk_level'].value_counts().sort_index())

print("\nMaximum probability:",
      geo_df['flood_probability'].max())

========== UPDATED RISK DISTRIBUTION ==========
risk_level
Low         1937
Moderate     574
High          49
Name: count, dtype: int64

Maximum probability: 0.5266666666666666


In [40]:
high_risk = risk_gdf.copy()
high_risk['risk_level'] = geo_df['risk_level'].values

high_risk = high_risk[
    high_risk['risk_level'] == 'High'
].copy()

print("High-risk locations:", len(high_risk))

print("\nHigh-risk probability range:")
if len(high_risk) > 0:
    print(
        high_risk['flood_probability'].min(),
        "to",
        high_risk['flood_probability'].max()
    )

High-risk locations: 49

High-risk probability range:
0.35333333333333333 to 0.5266666666666666


In [41]:
high_risk_buffer = high_risk.copy()

# Approximate local spatial zone around each high-risk prediction point
high_risk_buffer['geometry'] = high_risk_buffer.geometry.buffer(0.05)

print("High-risk points:", len(high_risk_buffer))
print("High-risk zone geometry created.")

High-risk points: 49
High-risk zone geometry created.


/tmp/ipykernel_513/2597756368.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  high_risk_buffer['geometry'] = high_risk_buffer.geometry.buffer(0.05)


In [42]:
roads_at_risk = gpd.sjoin(
    roads_gdf,
    high_risk_buffer[['geometry']],
    predicate='intersects',
    how='inner'
)

roads_at_risk_unique = roads_at_risk.drop_duplicates(
    subset=['system:index']
)

print("Road matches:", len(roads_at_risk))
print("Unique roads near high-risk zones:",
      len(roads_at_risk_unique))

print(
    "Road exposure percentage:",
    round(
        len(roads_at_risk_unique) / len(roads_gdf) * 100,
        2
    ),
    "%"
)

Road matches: 44700
Unique roads near high-risk zones: 6055
Road exposure percentage: 6.94 %


In [43]:
power_at_risk = gpd.sjoin(
    power_gdf,
    high_risk_buffer[['geometry']],
    predicate='intersects',
    how='inner'
)

power_at_risk_unique = power_at_risk.drop_duplicates(
    subset=['system:index']
)

print("Power matches:", len(power_at_risk))
print("Unique power infrastructure near high-risk zones:",
      len(power_at_risk_unique))

if 'infrastructure_type' in power_at_risk_unique.columns:
    print("\nPower infrastructure breakdown:")
    print(
        power_at_risk_unique[
            'infrastructure_type'
        ].value_counts()
    )

Power matches: 1653
Unique power infrastructure near high-risk zones: 653

Power infrastructure breakdown:
infrastructure_type
power_grid     652
power_plant      1
Name: count, dtype: int64


In [44]:
# Reproject to Web Mercator so buffering is performed in metres
high_risk_metric = high_risk.to_crs(epsg=3857)
roads_metric = roads_gdf.to_crs(epsg=3857)
power_metric = power_gdf.to_crs(epsg=3857)

# 0.05 degrees was roughly a few km; use 5 km as the
# explicit prototype exposure radius.
high_risk_metric['geometry'] = high_risk_metric.geometry.buffer(5000)

print("High-risk points:", len(high_risk_metric))
print("Exposure radius: 5 km")

High-risk points: 49
Exposure radius: 5 km


In [45]:
roads_at_risk_metric = gpd.sjoin(
    roads_metric,
    high_risk_metric[['geometry']],
    predicate='intersects',
    how='inner'
)

roads_at_risk_unique = roads_at_risk_metric.drop_duplicates(
    subset=['system:index']
)

road_exposure_pct = (
    len(roads_at_risk_unique) / len(roads_metric) * 100
)

print("Unique roads near high-risk zones:",
      len(roads_at_risk_unique))

print("Road exposure:",
      round(road_exposure_pct, 2), "%")

Unique roads near high-risk zones: 5519
Road exposure: 6.32 %


In [46]:
power_at_risk_metric = gpd.sjoin(
    power_metric,
    high_risk_metric[['geometry']],
    predicate='intersects',
    how='inner'
)

power_at_risk_unique = power_at_risk_metric.drop_duplicates(
    subset=['system:index']
)

power_exposure_pct = (
    len(power_at_risk_unique) / len(power_metric) * 100
)

print("Unique power infrastructure near high-risk zones:",
      len(power_at_risk_unique))

print("Power exposure:",
      round(power_exposure_pct, 2), "%")

if 'infrastructure_type' in power_at_risk_unique.columns:
    print("\nBreakdown:")
    print(
        power_at_risk_unique[
            'infrastructure_type'
        ].value_counts()
    )

Unique power infrastructure near high-risk zones: 553
Power exposure: 1.41 %

Breakdown:
infrastructure_type
power_grid     552
power_plant      1
Name: count, dtype: int64


In [47]:
final_risk_metrics = {
    "prediction_locations": len(geo_df),
    "high_risk_locations": len(high_risk),
    "moderate_risk_locations": int(
        (geo_df['risk_level'] == 'Moderate').sum()
    ),
    "low_risk_locations": int(
        (geo_df['risk_level'] == 'Low').sum()
    ),
    "mean_flood_probability": round(
        geo_df['flood_probability'].mean(), 4
    ),
    "maximum_flood_probability": round(
        geo_df['flood_probability'].max(), 4
    ),
    "high_risk_probability_min": round(
        high_risk['flood_probability'].min(), 4
    ),
    "high_risk_probability_max": round(
        high_risk['flood_probability'].max(), 4
    )
}

print(final_risk_metrics)

{'prediction_locations': 2560, 'high_risk_locations': 49, 'moderate_risk_locations': 574, 'low_risk_locations': 1937, 'mean_flood_probability': np.float64(0.1635), 'maximum_flood_probability': 0.5267, 'high_risk_probability_min': 0.3533, 'high_risk_probability_max': 0.5267}


In [48]:
final_infrastructure_metrics = {
    "total_roads": len(roads_gdf),
    "roads_near_high_risk": len(roads_at_risk_unique),
    "road_exposure_percent": round(road_exposure_pct, 2),

    "total_power_infrastructure": len(power_gdf),
    "power_near_high_risk": len(power_at_risk_unique),
    "power_exposure_percent": round(power_exposure_pct, 2),

    "power_grid_near_high_risk": int(
        (
            power_at_risk_unique['infrastructure_type']
            == 'power_grid'
        ).sum()
    ),
    "power_plants_near_high_risk": int(
        (
            power_at_risk_unique['infrastructure_type']
            == 'power_plant'
        ).sum()
    ),

    "exposure_radius_km": 5
}

print(final_infrastructure_metrics)

{'total_roads': 87306, 'roads_near_high_risk': 5519, 'road_exposure_percent': 6.32, 'total_power_infrastructure': 39168, 'power_near_high_risk': 553, 'power_exposure_percent': 1.41, 'power_grid_near_high_risk': 552, 'power_plants_near_high_risk': 1, 'exposure_radius_km': 5}


In [49]:
gemini_context = f"""
CYCLONE IMPACT & INFRASTRUCTURE VULNERABILITY ANALYSIS

MODEL OUTPUT
- Prediction locations: {final_risk_metrics['prediction_locations']}
- High-risk locations: {final_risk_metrics['high_risk_locations']}
- Moderate-risk locations: {final_risk_metrics['moderate_risk_locations']}
- Low-risk locations: {final_risk_metrics['low_risk_locations']}
- Mean RF flood-risk score: {final_risk_metrics['mean_flood_probability']}
- Maximum RF flood-risk score: {final_risk_metrics['maximum_flood_probability']}

HIGH-RISK LOCATIONS
- RF score range: {final_risk_metrics['high_risk_probability_min']} to {final_risk_metrics['high_risk_probability_max']}
- High-risk classification threshold used: 0.35
- Exposure analysis radius: 5 km

INFRASTRUCTURE NEAR HIGH-RISK LOCATIONS
- Total road features: {final_infrastructure_metrics['total_roads']}
- Roads near high-risk locations: {final_infrastructure_metrics['roads_near_high_risk']}
- Road exposure: {final_infrastructure_metrics['road_exposure_percent']}%

- Total power infrastructure features: {final_infrastructure_metrics['total_power_infrastructure']}
- Power infrastructure near high-risk locations: {final_infrastructure_metrics['power_near_high_risk']}
- Power exposure: {final_infrastructure_metrics['power_exposure_percent']}%
- Power-grid features near high-risk locations: {final_infrastructure_metrics['power_grid_near_high_risk']}
- Power plants near high-risk locations: {final_infrastructure_metrics['power_plants_near_high_risk']}

IMPORTANT:
These are model-derived risk and spatial-proximity results.
They indicate infrastructure located near predicted high-risk areas.
They do NOT establish that infrastructure was actually damaged or flooded.
"""

print(gemini_context)


CYCLONE IMPACT & INFRASTRUCTURE VULNERABILITY ANALYSIS

MODEL OUTPUT
- Prediction locations: 2560
- High-risk locations: 49
- Moderate-risk locations: 574
- Low-risk locations: 1937
- Mean RF flood-risk score: 0.1635
- Maximum RF flood-risk score: 0.5267

HIGH-RISK LOCATIONS
- RF score range: 0.3533 to 0.5267
- High-risk classification threshold used: 0.35
- Exposure analysis radius: 5 km

INFRASTRUCTURE NEAR HIGH-RISK LOCATIONS
- Total road features: 87306
- Roads near high-risk locations: 5519
- Road exposure: 6.32%

- Total power infrastructure features: 39168
- Power infrastructure near high-risk locations: 553
- Power exposure: 1.41%
- Power-grid features near high-risk locations: 552
- Power plants near high-risk locations: 1

IMPORTANT:
These are model-derived risk and spatial-proximity results.
They indicate infrastructure located near predicted high-risk areas.
They do NOT establish that infrastructure was actually damaged or flooded.



In [50]:
advisory_prompt = f"""
You are the AI advisory engine for a cyclone impact and infrastructure
vulnerability forecasting system.

Use ONLY the supplied analysis below. Do not invent infrastructure damage,
weather observations, locations, casualties, or evacuation orders.

{gemini_context}

Generate a concise emergency-planning advisory with exactly these sections:

1. RISK OVERVIEW
2. INFRASTRUCTURE EXPOSURE
3. PRIORITY ACTIONS
4. LIMITATIONS

Clearly distinguish model-predicted risk from confirmed real-world damage.
Use practical, actionable language suitable for a disaster-management dashboard.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=advisory_prompt
)

advisory = response.text

print(advisory)

**1. RISK OVERVIEW**
- Prediction locations evaluated: 2,560
- Risk classifications: 49 high-risk, 574 moderate-risk, 1,937 low-risk locations
- RF flood-risk scores: Mean 0.1635, Maximum 0.5267
- High-risk classification threshold: 0.35 (high-risk RF score range: 0.3533 to 0.5267 within a 5 km exposure analysis radius)

**2. INFRASTRUCTURE EXPOSURE**
- **Roads:** 5,519 out of 87,306 total road features (6.32%) are located near high-risk locations.
- **Power Infrastructure:** 553 out of 39,168 total power infrastructure features (1.41%) are located near high-risk locations, comprising 552 power-grid features and 1 power plant.

**3. PRIORITY ACTIONS**
- Deploy inspection and monitoring teams to focus on the 5,519 road features and 553 power infrastructure elements located near the 49 high-risk locations.
- Prioritize reinforcement and contingency planning for the single identified power plant and surrounding power-grid features within high-risk zones.
- Establish traffic management and

In [51]:
print("Gemini advisory generated:", len(advisory), "characters")

# Save the advisory
with open('/content/gemini_advisory.txt', 'w', encoding='utf-8') as f:
    f.write(advisory)

print("Saved: gemini_advisory.txt")

Gemini advisory generated: 1344 characters
Saved: gemini_advisory.txt


In [52]:
# Build the final structured result for the backend
final_output = {
    "risk": final_risk_metrics,
    "infrastructure": final_infrastructure_metrics,
    "advisory": advisory
}

print("FINAL AI OUTPUT READY")
print("Risk metrics:", len(final_output["risk"]))
print("Infrastructure metrics:", len(final_output["infrastructure"]))
print("Advisory:", len(final_output["advisory"]), "characters")

FINAL AI OUTPUT READY
Risk metrics: 8
Infrastructure metrics: 9
Advisory: 1344 characters


In [53]:
import pandas as pd
import ast

roads_df = pd.read_csv("/content/cyclone_roads.csv")

print("Road dataset shape:", roads_df.shape)
print("Columns:")
print(roads_df.columns.tolist())
roads_df.head()

Road dataset shape: (87306, 12)
Columns:
['system:index', 'GP_RAV', 'GP_RCY', 'GP_REX', 'GP_RRG', 'GP_RSE', 'GP_RSI', 'GP_RSY', 'GP_RTP', 'Shape_Leng', 'gp_gripreg', '.geo']


,system:index,GP_RAV,GP_RCY,GP_REX,GP_RRG,GP_RSE,GP_RSI,GP_RSY,GP_RTP,Shape_Leng,gp_gripreg,.geo
0,000700000000000033c8,1,50,1,18,0,51,2006,4,0.000216,6,"{""type"":""LineString"",""coordinates"":[[90.498459..."
1,000700000000000033c9,1,50,1,18,0,51,2006,4,0.000020,6,"{""type"":""LineString"",""coordinates"":[[90.498668..."
2,000700000000000033ca,1,50,1,18,0,51,2006,4,0.000040,6,"{""type"":""LineString"",""coordinates"":[[90.498708..."
3,000700000000000033cc,1,50,1,18,0,51,2006,4,0.000902,6,"{""type"":""LineString"",""coordinates"":[[90.401259..."
4,000700000000000033cd,1,50,1,18,0,51,2006,4,0.000623,6,"{""type"":""LineString"",""coordinates"":[[90.403631..."


In [54]:
print("Example geometry:")
print(roads_df[".geo"].iloc[0])

Example geometry:
{"type":"LineString","coordinates":[[90.49845917742113,22.235838036033233],[90.49853944124008,22.2358603315385],[90.49866875517064,22.235891545245877]]}


In [55]:
print("Road attribute summary:")
print(roads_df[
    ["GP_RAV", "GP_RCY", "GP_REX", "GP_RRG", "GP_RSE", "GP_RSI", "GP_RSY", "GP_RTP", "Shape_Leng"]
].describe(include="all"))

print("\nMissing values:")
print(roads_df.isna().sum())

Road attribute summary:
             GP_RAV        GP_RCY   GP_REX   GP_RRG        GP_RSE  \
count  87306.000000  87306.000000  87306.0  87306.0  87306.000000   
mean       1.538588    210.938630      1.0     18.0      1.127987   
std        0.498512    152.794783      0.0      0.0      1.002133   
min        1.000000     50.000000      1.0     18.0      0.000000   
25%        1.000000     50.000000      1.0     18.0      0.000000   
50%        2.000000    356.000000      1.0     18.0      1.000000   
75%        2.000000    356.000000      1.0     18.0      2.000000   
max        2.000000    356.000000      1.0     18.0      3.000000   

             GP_RSI        GP_RSY        GP_RTP    Shape_Leng  
count  87306.000000  87306.000000  87306.000000  8.730600e+04  
mean      50.426088   2010.079594      4.332383  1.350278e-02  
std        7.977379      3.807943      0.750905  5.952699e-02  
min       22.000000   2006.000000      1.000000  4.560700e-08  
25%       51.000000   2006.000000 

In [56]:
import json
import geopandas as gpd
from shapely.geometry import shape

roads_df["geometry"] = roads_df[".geo"].apply(
    lambda x: shape(json.loads(x))
)

roads_gdf = gpd.GeoDataFrame(
    roads_df,
    geometry="geometry",
    crs="EPSG:4326"
)

print("Road GeoDataFrame:", roads_gdf.shape)
print("CRS:", roads_gdf.crs)
print("Geometry types:")
print(roads_gdf.geometry.geom_type.value_counts())

Road GeoDataFrame: (87306, 13)
CRS: EPSG:4326
Geometry types:
LineString         86984
LinearRing           156
MultiLineString      129
Point                 37
Name: count, dtype: int64


In [57]:
roads_projected = roads_gdf.to_crs("EPSG:3857")

roads_gdf["road_length_km"] = (
    roads_projected.geometry.length / 1000
)

print("Total road length:",
      round(roads_gdf["road_length_km"].sum(), 2),
      "km")

print("\nRoad length statistics:")
print(
    roads_gdf["road_length_km"].describe()
)

Total road length: 136617.74 km

Road length statistics:
count    87306.000000
mean         1.564815
std          6.828289
min          0.000000
25%          0.084820
50%          0.367929
75%          1.785478
max       1120.096659
Name: road_length_km, dtype: float64


In [58]:
routing_roads = roads_gdf[
    [
        "system:index",
        "GP_RAV",
        "GP_RCY",
        "GP_RSE",
        "GP_RSI",
        "GP_RTP",
        "road_length_km",
        "geometry"
    ]
].copy()

print("Routing road dataset:", routing_roads.shape)
routing_roads.head()

Routing road dataset: (87306, 8)


,system:index,GP_RAV,GP_RCY,GP_RSE,GP_RSI,GP_RTP,road_length_km,geometry
0,000700000000000033c8,1,50,0,51,4,0.024205,"LINESTRING (90.49846 22.23584, 90.49854 22.235..."
1,000700000000000033c9,1,50,0,51,4,0.001986,"LINESTRING (90.49867 22.23589, 90.49865 22.23589)"
2,000700000000000033ca,1,50,0,51,4,0.004467,"LINESTRING (90.49871 22.23589, 90.49867 22.23589)"
3,000700000000000033cc,1,50,0,51,4,0.108595,"LINESTRING (90.40126 22.29666, 90.4013 22.2960..."
4,000700000000000033cd,1,50,0,51,4,0.074853,"LINESTRING (90.40363 22.29666, 90.40367 22.296..."


In [59]:
# Keep only actual line geometries for routing
routing_roads = routing_roads[
    routing_roads.geometry.geom_type.isin(
        ["LineString", "MultiLineString"]
    )
].copy()

print("Roads after geometry filtering:", len(routing_roads))
print(routing_roads.geometry.geom_type.value_counts())

Roads after geometry filtering: 87113
LineString         86984
MultiLineString      129
Name: count, dtype: int64


In [60]:
# Convert MultiLineString roads into individual LineStrings
routing_roads = routing_roads.explode(
    index_parts=False
).reset_index(drop=True)

print("Road segments after exploding MultiLineStrings:", len(routing_roads))
print(routing_roads.geometry.geom_type.value_counts())

Road segments after exploding MultiLineStrings: 87307
LineString    87307
Name: count, dtype: int64


In [61]:
# Remove zero-length geometries
routing_roads["segment_length_km"] = (
    routing_roads.to_crs("EPSG:3857").geometry.length / 1000
)

routing_roads = routing_roads[
    routing_roads["segment_length_km"] > 0
].copy()

print("Final routing segments:", len(routing_roads))
print(
    "Total routing length:",
    round(routing_roads["segment_length_km"].sum(), 2),
    "km"
)

Final routing segments: 87307
Total routing length: 136426.87 km


In [62]:
# Convert the RF prediction locations into a GeoDataFrame

prediction_points = gpd.GeoDataFrame(
    geo_df.copy(),
    geometry=gpd.points_from_xy(
        geo_df["longitude"],
        geo_df["latitude"]
    ),
    crs="EPSG:4326"
)

print("Prediction points:", len(prediction_points))
print(
    prediction_points[
        ["latitude", "longitude", "flood_probability", "flood_prediction"]
    ].head()
)

Prediction points: 2560
    latitude  longitude  flood_probability  flood_prediction
0  21.784294  87.817810           0.243333                 0
1  22.113557  88.226040           0.116667                 0
2  21.926739  87.863489           0.116667                 0
3  22.184625  88.254728           0.196667                 0
4  22.153667  88.453711           0.150000                 0


In [63]:
# Use a metre-based CRS for the nearest-neighbour operation

routing_roads_proj = routing_roads.to_crs("EPSG:3857")
prediction_points_proj = prediction_points.to_crs("EPSG:3857")

print("Road CRS:", routing_roads_proj.crs)
print("Prediction CRS:", prediction_points_proj.crs)

Road CRS: EPSG:3857
Prediction CRS: EPSG:3857


In [64]:
# Use the midpoint of each road segment as its representative location

routing_roads_proj["road_midpoint"] = (
    routing_roads_proj.geometry.interpolate(0.5, normalized=True)
)

road_midpoints = gpd.GeoDataFrame(
    routing_roads_proj.drop(columns="geometry"),
    geometry="road_midpoint",
    crs=routing_roads_proj.crs
)

road_risk = gpd.sjoin_nearest(
    road_midpoints,
    prediction_points_proj[
        ["flood_probability", "flood_prediction", "geometry"]
    ],
    how="left",
    distance_col="risk_point_distance_m"
)

print("Roads with risk:", len(road_risk))
print(
    road_risk[
        [
            "flood_probability",
            "flood_prediction",
            "risk_point_distance_m"
        ]
    ].head()
)

Roads with risk: 87307
   flood_probability  flood_prediction  risk_point_distance_m
0           0.113333                 0          222788.740685
1           0.113333                 0          222799.583880
2           0.113333                 0          222802.807199
3           0.260000                 0          211957.039941
4           0.260000                 0          212219.672203


In [65]:
print("Risk-point distance statistics (km):")

print(
    (road_risk["risk_point_distance_m"] / 1000).describe()
)

print("\nDistance percentiles (km):")

print(
    (road_risk["risk_point_distance_m"] / 1000)
    .quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

Risk-point distance statistics (km):
count    87307.000000
mean       128.205099
std         94.853438
min          0.023059
25%         25.000627
50%        139.653500
75%        206.403501
max       1158.100137
Name: risk_point_distance_m, dtype: float64

Distance percentiles (km):
0.25     25.000627
0.50    139.653500
0.75    206.403501
0.90    259.290321
0.95    269.433173
0.99    308.814613
Name: risk_point_distance_m, dtype: float64


In [66]:
MAX_RISK_DISTANCE_KM = 10

road_risk["risk_distance_km"] = (
    road_risk["risk_point_distance_m"] / 1000
)

road_risk_valid = road_risk[
    road_risk["risk_distance_km"] <= MAX_RISK_DISTANCE_KM
].copy()

print("Road segments with valid risk linkage:",
      len(road_risk_valid))

print(
    "Coverage:",
    round(
        len(road_risk_valid) / len(road_risk) * 100,
        2
    ),
    "%"
)

Road segments with valid risk linkage: 7318
Coverage: 8.38 %


In [67]:
print("Road flood-probability statistics:")

print(
    road_risk_valid["flood_probability"].describe()
)

print("\nRoad flood-risk categories:")

road_risk_valid["road_risk_category"] = pd.cut(
    road_risk_valid["flood_probability"],
    bins=[-0.01, 0.15, 0.35, 1.0],
    labels=["Low", "Moderate", "High"]
)

print(
    road_risk_valid["road_risk_category"]
    .value_counts()
)

Road flood-probability statistics:
count    7318.000000
mean        0.325261
std         0.129062
min         0.046667
25%         0.183333
50%         0.326667
75%         0.483333
max         0.496667
Name: flood_probability, dtype: float64

Road flood-risk categories:
road_risk_category
Moderate    4050
High        2613
Low          655
Name: count, dtype: int64


In [68]:
print("Prediction point coverage:")
print(
    prediction_points[
        ["latitude", "longitude", "flood_probability"]
    ].describe()
)

print("\nPrediction point bounds:")
print(
    prediction_points.total_bounds
)

Prediction point coverage:
          latitude    longitude  flood_probability
count  2560.000000  2560.000000        2560.000000
mean     22.211720    88.115502           0.163512
std       0.179925     0.212025           0.070715
min      21.708160    87.800303           0.026667
25%      22.081255    87.920161           0.113333
50%      22.225211    88.094010           0.150000
75%      22.362342    88.305744           0.200000
max      22.500035    88.499407           0.526667

Prediction point bounds:
[87.80030252 21.70815995 88.49940737 22.50003505]


In [69]:
prediction_points_proj = prediction_points.to_crs("EPSG:3857")

nearest_prediction = gpd.sjoin_nearest(
    prediction_points_proj,
    prediction_points_proj[
        ["geometry"]
    ],
    how="left",
    distance_col="nearest_distance_m"
)

# Remove self-matches
nearest_prediction = nearest_prediction[
    nearest_prediction["nearest_distance_m"] > 0
]

print("Nearest prediction-point distance (km):")
print(
    (nearest_prediction["nearest_distance_m"] / 1000).describe()
)

Nearest prediction-point distance (km):
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: nearest_distance_m, dtype: float64


In [70]:
print("Prediction-point spacing percentiles (km):")

print(
    (
        nearest_prediction["nearest_distance_m"] / 1000
    ).quantile(
        [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

Prediction-point spacing percentiles (km):
0.10   NaN
0.25   NaN
0.50   NaN
0.75   NaN
0.90   NaN
0.95   NaN
0.99   NaN
Name: nearest_distance_m, dtype: float64


In [71]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

prediction_points_proj = prediction_points.to_crs("EPSG:3857")

# Extract projected coordinates
coords = np.column_stack([
    prediction_points_proj.geometry.x,
    prediction_points_proj.geometry.y
])

# Find the 2 nearest points:
# 1st = the point itself
# 2nd = nearest DIFFERENT prediction point
nn = NearestNeighbors(
    n_neighbors=2,
    metric="euclidean"
)

nn.fit(coords)

distances, indices = nn.kneighbors(coords)

nearest_other_distance_km = distances[:, 1] / 1000

print("Nearest OTHER prediction-point distance (km):")
print(
    pd.Series(nearest_other_distance_km).describe()
)

Nearest OTHER prediction-point distance (km):
count    2560.000000
mean        0.701510
std         0.383097
min         0.019937
25%         0.419788
50%         0.640972
75%         0.952289
max         2.194196
dtype: float64


In [72]:
spacing_percentiles = pd.Series(
    nearest_other_distance_km
).quantile(
    [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

print("Prediction-point spacing percentiles (km):")
print(spacing_percentiles)

Prediction-point spacing percentiles (km):
0.10    0.251123
0.25    0.419788
0.50    0.640972
0.75    0.952289
0.90    1.212325
0.95    1.419400
0.99    1.785341
dtype: float64


In [73]:
print(
    "Minimum spacing:",
    round(nearest_other_distance_km.min(), 2),
    "km"
)

print(
    "Median spacing:",
    round(np.median(nearest_other_distance_km), 2),
    "km"
)

print(
    "Maximum nearest-neighbour spacing:",
    round(nearest_other_distance_km.max(), 2),
    "km"
)

Minimum spacing: 0.02 km
Median spacing: 0.64 km
Maximum nearest-neighbour spacing: 2.19 km


In [74]:
MAX_RISK_DISTANCE_KM = 3

road_risk = gpd.sjoin_nearest(
    road_midpoints,
    prediction_points_proj[
        ["flood_probability", "flood_prediction", "geometry"]
    ],
    how="left",
    distance_col="risk_point_distance_m"
)

road_risk["risk_distance_km"] = (
    road_risk["risk_point_distance_m"] / 1000
)

road_risk_valid = road_risk[
    road_risk["risk_distance_km"] <= MAX_RISK_DISTANCE_KM
].copy()

print("Total road segments:", len(road_risk))
print("Roads with valid risk linkage:", len(road_risk_valid))

print(
    "Coverage:",
    round(
        len(road_risk_valid) / len(road_risk) * 100,
        2
    ),
    "%"
)

Total road segments: 87307
Roads with valid risk linkage: 4998
Coverage: 5.72 %


In [75]:
print("Road-to-RF-point distance statistics (km):")

print(
    road_risk_valid["risk_distance_km"].describe()
)

print("\nDistance percentiles:")

print(
    road_risk_valid["risk_distance_km"].quantile(
        [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

Road-to-RF-point distance statistics (km):
count    4998.000000
mean        1.196241
std         0.743690
min         0.023059
25%         0.604062
50%         0.994714
75%         1.683062
max         2.999719
Name: risk_distance_km, dtype: float64

Distance percentiles:
0.25    0.604062
0.50    0.994714
0.75    1.683062
0.90    2.353143
0.95    2.715278
0.99    2.942321
Name: risk_distance_km, dtype: float64


In [76]:
road_risk_valid["road_risk_category"] = pd.cut(
    road_risk_valid["flood_probability"],
    bins=[-0.01, 0.15, 0.35, 1.0],
    labels=["Low", "Moderate", "High"]
)

print("Road risk distribution:")
print(
    road_risk_valid["road_risk_category"]
    .value_counts()
)

print("\nFlood probability statistics:")
print(
    road_risk_valid["flood_probability"].describe()
)

Road risk distribution:
road_risk_category
Moderate    2576
High        1796
Low          626
Name: count, dtype: int64

Flood probability statistics:
count    4998.000000
mean        0.314190
std         0.133746
min         0.053333
25%         0.163333
50%         0.326667
75%         0.440000
max         0.496667
Name: flood_probability, dtype: float64


In [77]:
print("=== ROAD NETWORK BOUNDS ===")
print(routing_roads.total_bounds)
print("(min_lon, min_lat, max_lon, max_lat)")

print("\n=== RF PREDICTION BOUNDS ===")
print(prediction_points.total_bounds)
print("(min_lon, min_lat, max_lon, max_lat)")

=== ROAD NETWORK BOUNDS ===
[77.39255474 19.1544253  90.58368152 27.87914256]
(min_lon, min_lat, max_lon, max_lat)

=== RF PREDICTION BOUNDS ===
[87.80030252 21.70815995 88.49940737 22.50003505]
(min_lon, min_lat, max_lon, max_lat)


In [78]:
prediction_bounds = prediction_points.total_bounds

roads_inside_prediction_extent = routing_roads[
    routing_roads.geometry.bounds["minx"] >= prediction_bounds[0]
].copy()

roads_inside_prediction_extent = roads_inside_prediction_extent[
    roads_inside_prediction_extent.geometry.bounds["maxx"] <= prediction_bounds[2]
]

roads_inside_prediction_extent = roads_inside_prediction_extent[
    roads_inside_prediction_extent.geometry.bounds["miny"] >= prediction_bounds[1]
]

roads_inside_prediction_extent = roads_inside_prediction_extent[
    roads_inside_prediction_extent.geometry.bounds["maxy"] <= prediction_bounds[3]
]

print(
    "Road segments inside RF prediction bounding box:",
    len(roads_inside_prediction_extent)
)

print(
    "Percentage of total roads:",
    round(
        len(roads_inside_prediction_extent) /
        len(routing_roads) * 100,
        2
    ),
    "%"
)

Road segments inside RF prediction bounding box: 3899
Percentage of total roads: 4.47 %


In [79]:
print("RF prediction points by latitude:")
print(
    prediction_points["latitude"].describe()
)

print("\nRF prediction points by longitude:")
print(
    prediction_points["longitude"].describe()
)

print("\nRoad network coordinate range:")
print(
    "Longitude:",
    round(routing_roads.total_bounds[0], 3),
    "to",
    round(routing_roads.total_bounds[2], 3)
)

print(
    "Latitude:",
    round(routing_roads.total_bounds[1], 3),
    "to",
    round(routing_roads.total_bounds[3], 3)
)

RF prediction points by latitude:
count    2560.000000
mean       22.211720
std         0.179925
min        21.708160
25%        22.081255
50%        22.225211
75%        22.362342
max        22.500035
Name: latitude, dtype: float64

RF prediction points by longitude:
count    2560.000000
mean       88.115502
std         0.212025
min        87.800303
25%        87.920161
50%        88.094010
75%        88.305744
max        88.499407
Name: longitude, dtype: float64

Road network coordinate range:
Longitude: 77.393 to 90.584
Latitude: 19.154 to 27.879


In [80]:
# Use the RF prediction extent as the operational forecast region

routing_region = prediction_points.unary_union.convex_hull

print("Routing region created.")
print("Area (approx):",
      round(routing_region.area / 1e6, 2),
      "km²")

Routing region created.
Area (approx): 0.0 km²


/tmp/ipykernel_513/3146830382.py:3: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  routing_region = prediction_points.unary_union.convex_hull


In [81]:
# Work in the projected CRS for spatial operations

routing_roads_proj = routing_roads.to_crs("EPSG:3857")

routing_region_proj = gpd.GeoSeries(
    [routing_region],
    crs="EPSG:4326"
).to_crs("EPSG:3857").iloc[0]

roads_in_forecast_region = routing_roads_proj[
    routing_roads_proj.geometry.intersects(routing_region_proj)
].copy()

print(
    "Road segments intersecting forecast region:",
    len(roads_in_forecast_region)
)

print(
    "Percentage of total road network:",
    round(
        len(roads_in_forecast_region) /
        len(routing_roads_proj) * 100,
        2
    ),
    "%"
)

Road segments intersecting forecast region: 3306
Percentage of total road network: 3.79 %


In [82]:
roads_in_forecast_region_wgs84 = (
    roads_in_forecast_region
    .to_crs("EPSG:4326")
)

print("Forecast-region road bounds:")
print(
    roads_in_forecast_region_wgs84.total_bounds
)

print("\nRoad count by road class:")
print(
    roads_in_forecast_region_wgs84["GP_RTP"]
    .value_counts()
)

Forecast-region road bounds:
[81.56660776 21.16681315 88.56506431 22.60724102]

Road count by road class:
GP_RTP
5    2891
4     351
3      36
2      27
1       1
Name: count, dtype: int64


In [85]:
min_lon, min_lat, max_lon, max_lat = prediction_points.total_bounds

forecast_box = gpd.GeoSeries(
    [
        shapely.geometry.box(
            min_lon,
            min_lat,
            max_lon,
            max_lat
        )
    ],
    crs="EPSG:4326"
)

forecast_box_proj = forecast_box.to_crs("EPSG:3857").iloc[0]

print("Forecast box:")
print(
    min_lon,
    min_lat,
    max_lon,
    max_lat
)

Forecast box:
87.80030252114653 21.708159948017993 88.49940737165434 22.50003504734515


In [84]:
import shapely

In [86]:
roads_in_forecast_region = routing_roads_proj[
    routing_roads_proj.geometry.intersects(
        forecast_box_proj
    )
].copy()

print(
    "Road segments in RF forecast region:",
    len(roads_in_forecast_region)
)

print(
    "Coverage of original road dataset:",
    round(
        len(roads_in_forecast_region) /
        len(routing_roads_proj) * 100,
        2
    ),
    "%"
)


Road segments in RF forecast region: 4022
Coverage of original road dataset: 4.61 %


In [87]:
road_midpoints_region = (
    roads_in_forecast_region
    .geometry
    .interpolate(0.5, normalized=True)
)

road_midpoints_region = gpd.GeoDataFrame(
    roads_in_forecast_region.drop(
        columns=["geometry"]
    ),
    geometry=road_midpoints_region,
    crs="EPSG:3857"
)

road_risk_region = gpd.sjoin_nearest(
    road_midpoints_region,
    prediction_points_proj[
        [
            "flood_probability",
            "flood_prediction",
            "geometry"
        ]
    ],
    how="left",
    distance_col="risk_point_distance_m"
)

road_risk_region["risk_distance_km"] = (
    road_risk_region["risk_point_distance_m"] / 1000
)

print(
    "Roads with RF risk:",
    len(road_risk_region)
)

print("\nRisk-distance statistics:")
print(
    road_risk_region["risk_distance_km"].describe()
)

Roads with RF risk: 4022

Risk-distance statistics:
count    4022.000000
mean        1.227146
std         5.241526
min         0.023059
25%         0.536552
50%         0.829319
75%         1.411740
max       317.481909
Name: risk_distance_km, dtype: float64


In [88]:
print("=== ROAD-RISK DATASET ===")

print("Road segments:", len(road_risk_region))

print("\nRisk distance (km):")
print(
    road_risk_region["risk_distance_km"].describe()
)

print("\nFlood probability:")
print(
    road_risk_region["flood_probability"].describe()
)

=== ROAD-RISK DATASET ===
Road segments: 4022

Risk distance (km):
count    4022.000000
mean        1.227146
std         5.241526
min         0.023059
25%         0.536552
50%         0.829319
75%         1.411740
max       317.481909
Name: risk_distance_km, dtype: float64

Flood probability:
count    4022.000000
mean        0.314012
std         0.137381
min         0.053333
25%         0.163333
50%         0.316667
75%         0.440000
max         0.496667
Name: flood_probability, dtype: float64


In [89]:
road_risk_region["road_risk_category"] = pd.cut(
    road_risk_region["flood_probability"],
    bins=[-0.01, 0.15, 0.35, 1.0],
    labels=["Low", "Moderate", "High"]
)

print("Road-risk distribution:")
print(
    road_risk_region["road_risk_category"]
    .value_counts()
)

Road-risk distribution:
road_risk_category
Moderate    1802
High        1568
Low          652
Name: count, dtype: int64


In [90]:
routing_risk = road_risk_region[
    [
        "system:index",
        "GP_RAV",
        "GP_RCY",
        "GP_RSE",
        "GP_RSI",
        "GP_RTP",
        "road_length_km",
        "flood_probability",
        "flood_prediction",
        "road_risk_category",
        "risk_distance_km",
        "geometry"
    ]
].copy()

print("Final routing-risk dataset:", routing_risk.shape)
routing_risk.head()

Final routing-risk dataset: (4022, 12)


,system:index,GP_RAV,GP_RCY,GP_RSE,GP_RSI,GP_RTP,road_length_km,flood_probability,flood_prediction,road_risk_category,risk_distance_km,geometry
58573,003c0000000000003b93,2,356,1,22,2,64.092422,0.100000,0,Low,1.261837,POINT (9788516.758 2533112.989)
58691,003c000000000000665e,2,356,1,22,3,65.256039,0.220000,0,Moderate,0.915164,POINT (9792790.732 2533724.205)
58692,003c000000000000665f,2,356,1,22,3,109.332001,0.076667,0,Low,3.161623,POINT (9819477.327 2509034.948)
58694,003c0000000000003c9f,1,356,3,22,4,4.805023,0.093333,0,Low,1.204739,POINT (9806092.896 2536454.946)
58695,003c0000000000003ca0,1,356,3,22,4,7.904241,0.053333,0,Low,0.534858,POINT (9812107.336 2538393.135)


In [91]:
# Cyclone timeline
cyclone_timeline = pd.DataFrame({
    "time_to_landfall_h": [
        48, 42, 36, 30, 24, 18, 12, 6, 0
    ],
    "wind_kmh": [
        65, 80, 95, 110, 125, 135, 145, 135, 115
    ]
})

cyclone_timeline

,time_to_landfall_h,wind_kmh
0,48,65
1,42,80
2,36,95
3,30,110
4,24,125
5,18,135
6,12,145
7,6,135
8,0,115


In [92]:
# Normalized cyclone intensity factor
#
# 0 = weakest point in our timeline
# 1 = strongest point in our timeline

cyclone_timeline["intensity_factor"] = (
    cyclone_timeline["wind_kmh"] /
    cyclone_timeline["wind_kmh"].max()
)

# Convert time-to-landfall into an approach factor
#
# 0 = 48 hours away
# 1 = landfall

cyclone_timeline["approach_factor"] = (
    1 -
    cyclone_timeline["time_to_landfall_h"] /
    cyclone_timeline["time_to_landfall_h"].max()
)

print(cyclone_timeline)

   time_to_landfall_h  wind_kmh  intensity_factor  approach_factor
0                  48        65          0.448276            0.000
1                  42        80          0.551724            0.125
2                  36        95          0.655172            0.250
3                  30       110          0.758621            0.375
4                  24       125          0.862069            0.500
5                  18       135          0.931034            0.625
6                  12       145          1.000000            0.750
7                   6       135          0.931034            0.875
8                   0       115          0.793103            1.000


In [93]:
# Create a road degradation score for every time step.
#
# This is a prototype operational score, not a physical
# flood-depth prediction.

road_base_risk = routing_risk["flood_probability"].fillna(0)

for _, row in cyclone_timeline.iterrows():

    t = int(row["time_to_landfall_h"])

    intensity = row["intensity_factor"]
    approach = row["approach_factor"]

    # Increasing degradation as cyclone approaches
    degradation = (
        road_base_risk *
        (0.5 + 0.5 * intensity) *
        (0.5 + 0.5 * approach)
    )

    routing_risk[f"degradation_Tminus_{t}h"] = degradation

print("Created time-dependent degradation columns:")
print([
    c for c in routing_risk.columns
    if "degradation" in c
])

Created time-dependent degradation columns:
['degradation_Tminus_48h', 'degradation_Tminus_42h', 'degradation_Tminus_36h', 'degradation_Tminus_30h', 'degradation_Tminus_24h', 'degradation_Tminus_18h', 'degradation_Tminus_12h', 'degradation_Tminus_6h', 'degradation_Tminus_0h']


In [94]:
degradation_columns = [
    c for c in routing_risk.columns
    if c.startswith("degradation_Tminus_")
]

degradation_summary = routing_risk[
    degradation_columns
].mean().sort_index()

print("Mean road degradation at each time:")
print(degradation_summary)

Mean road degradation at each time:
degradation_Tminus_0h     0.281528
degradation_Tminus_12h    0.274761
degradation_Tminus_18h    0.246337
degradation_Tminus_24h    0.219267
degradation_Tminus_30h    0.189828
degradation_Tminus_36h    0.162420
degradation_Tminus_42h    0.137042
degradation_Tminus_48h    0.113694
degradation_Tminus_6h     0.284235
dtype: float64


In [95]:
sample_road = routing_risk.iloc[0]

print("Sample road flood probability:",
      round(sample_road["flood_probability"], 4))

print("\nDegradation over time:")

for col in degradation_columns:
    print(
        col,
        "→",
        round(sample_road[col], 4)
    )

Sample road flood probability: 0.1

Degradation over time:
degradation_Tminus_48h → 0.0362
degradation_Tminus_42h → 0.0436
degradation_Tminus_36h → 0.0517
degradation_Tminus_30h → 0.0605
degradation_Tminus_24h → 0.0698
degradation_Tminus_18h → 0.0784
degradation_Tminus_12h → 0.0875
degradation_Tminus_6h → 0.0905
degradation_Tminus_0h → 0.0897


In [96]:
for col in degradation_columns:

    hours = col.split("_")[-1].replace("h", "")

    routing_risk[f"routing_cost_Tminus_{hours}h"] = (
        routing_risk["road_length_km"] *
        (
            1 +
            10 * routing_risk[col]
        )
    )

cost_columns = [
    c for c in routing_risk.columns
    if c.startswith("routing_cost_")
]

print("Routing cost columns:")
print(cost_columns)

Routing cost columns:
['routing_cost_Tminus_48h', 'routing_cost_Tminus_42h', 'routing_cost_Tminus_36h', 'routing_cost_Tminus_30h', 'routing_cost_Tminus_24h', 'routing_cost_Tminus_18h', 'routing_cost_Tminus_12h', 'routing_cost_Tminus_6h', 'routing_cost_Tminus_0h']


In [97]:
import networkx as nx
from shapely.geometry import LineString

# Work with the forecast-region roads
graph_roads = routing_risk.copy()

# Ensure LineString geometry
graph_roads = graph_roads[
    graph_roads.geometry.geom_type == "LineString"
].copy()

# Create graph
road_graph = nx.Graph()

for idx, row in graph_roads.iterrows():

    coords = list(row.geometry.coords)

    start = coords[0]
    end = coords[-1]

    road_graph.add_edge(
        start,
        end,
        road_index=idx,
        length_km=row["road_length_km"],
        flood_probability=row["flood_probability"]
    )

print("Road graph created")
print("Nodes:", road_graph.number_of_nodes())
print("Edges:", road_graph.number_of_edges())

Road graph created
Nodes: 0
Edges: 0


In [98]:
for u, v, data in road_graph.edges(data=True):

    idx = data["road_index"]

    for t in [48, 42, 36, 30, 24, 18, 12, 6, 0]:

        cost_col = f"routing_cost_Tminus_{t}h"

        if cost_col in routing_risk.columns:
            data[cost_col] = routing_risk.loc[
                idx, cost_col
            ]

print("Dynamic routing costs attached.")

Dynamic routing costs attached.


In [99]:
print("=== ROAD GRAPH ===")
print("Nodes:", road_graph.number_of_nodes())
print("Edges:", road_graph.number_of_edges())

sample_edges = list(road_graph.edges(data=True))[:5]

for u, v, data in sample_edges:
    print("\nEdge:")
    print("Start:", u)
    print("End:", v)
    print("Length:", round(data["length_km"], 3), "km")
    print("Flood probability:",
          round(data["flood_probability"], 4))

=== ROAD GRAPH ===
Nodes: 0
Edges: 0


In [100]:
print("routing_risk shape:", routing_risk.shape)
print("Geometry column:", routing_risk.geometry.name)
print("Geometry types:")
print(routing_risk.geometry.geom_type.value_counts())

print("\nMissing geometries:", routing_risk.geometry.isna().sum())
print("\nSample geometry:")
print(routing_risk.geometry.iloc[0])

routing_risk shape: (4022, 30)
Geometry column: geometry
Geometry types:
Point    4022
Name: count, dtype: int64

Missing geometries: 0

Sample geometry:
POINT (9788516.757508587 2533112.9889557865)


In [101]:
graph_roads = routing_risk.copy()

# Keep only valid line geometries
graph_roads = graph_roads[
    graph_roads.geometry.notna()
].copy()

graph_roads = graph_roads[
    graph_roads.geometry.geom_type.isin(
        ["LineString", "MultiLineString"]
    )
].copy()

# Convert MultiLineString → individual LineStrings
graph_roads = graph_roads.explode(
    index_parts=False
).reset_index(drop=True)

print("Graph road segments:", len(graph_roads))
print(graph_roads.geometry.geom_type.value_counts())

Graph road segments: 0
Series([], Name: count, dtype: int64)


In [102]:
road_graph = nx.Graph()

for idx, row in graph_roads.iterrows():

    coords = list(row.geometry.coords)

    start = tuple(coords[0])
    end = tuple(coords[-1])

    if start == end:
        continue

    road_graph.add_edge(
        start,
        end,
        road_index=idx,
        length_km=float(row["segment_length_km"]),
        flood_probability=float(row["flood_probability"])
    )

print("=== ROAD GRAPH ===")
print("Nodes:", road_graph.number_of_nodes())
print("Edges:", road_graph.number_of_edges())

=== ROAD GRAPH ===
Nodes: 0
Edges: 0


In [103]:
# Rebuild routing dataset while restoring the ORIGINAL road geometries

graph_roads = roads_in_forecast_region.copy()

# Keep only actual road line geometries
graph_roads = graph_roads[
    graph_roads.geometry.notna() &
    graph_roads.geometry.geom_type.isin(
        ["LineString", "MultiLineString"]
    )
].copy()

# Explode MultiLineStrings
graph_roads = (
    graph_roads
    .explode(index_parts=False)
    .reset_index(drop=True)
)

# Add the road-risk information from the midpoint spatial join
risk_info = road_risk_region[
    [
        "system:index",
        "flood_probability",
        "flood_prediction",
        "road_risk_category",
        "risk_distance_km"
    ]
].copy()

graph_roads = graph_roads.merge(
    risk_info,
    on="system:index",
    how="inner"
)

print("=== GRAPH ROAD DATA ===")
print("Road segments:", len(graph_roads))
print("Geometry types:")
print(graph_roads.geometry.geom_type.value_counts())
print("\nColumns:")
print(graph_roads.columns.tolist())

=== GRAPH ROAD DATA ===
Road segments: 4024
Geometry types:
LineString    4024
Name: count, dtype: int64

Columns:
['system:index', 'GP_RAV', 'GP_RCY', 'GP_RSE', 'GP_RSI', 'GP_RTP', 'road_length_km', 'segment_length_km', 'geometry', 'flood_probability', 'flood_prediction', 'road_risk_category', 'risk_distance_km']


In [104]:
road_graph = nx.Graph()

for idx, row in graph_roads.iterrows():

    coords = list(row.geometry.coords)

    start = tuple(coords[0])
    end = tuple(coords[-1])

    if start == end:
        continue

    road_graph.add_edge(
        start,
        end,
        road_index=idx,
        length_km=float(row["segment_length_km"]),
        flood_probability=float(row["flood_probability"]),
        road_risk_category=str(row["road_risk_category"])
    )

print("=== ROAD NETWORK ===")
print("Nodes:", road_graph.number_of_nodes())
print("Edges:", road_graph.number_of_edges())

=== ROAD NETWORK ===
Nodes: 3953
Edges: 3993


In [105]:
# Map each graph edge to its corresponding routing-risk row

for u, v, data in road_graph.edges(data=True):

    idx = data["road_index"]

    for t in [48, 42, 36, 30, 24, 18, 12, 6, 0]:

        data[f"cost_{t}h"] = float(
            graph_roads.loc[idx, f"routing_cost_Tminus_{t}h"]
        )

print("Dynamic costs attached to graph edges.")

KeyError: 'routing_cost_Tminus_48h'

In [106]:
# Keep the original road geometry + ALL dynamic routing costs

cost_columns = [
    f"routing_cost_Tminus_{t}h"
    for t in [48, 42, 36, 30, 24, 18, 12, 6, 0]
]

risk_columns = [
    "system:index",
    "flood_probability",
    "flood_prediction",
    "road_risk_category",
    "risk_distance_km"
]

graph_roads = graph_roads.merge(
    routing_risk[
        risk_columns + cost_columns
    ],
    on="system:index",
    how="left",
    suffixes=("", "_risk")
)

print("Graph roads:", len(graph_roads))
print("Dynamic cost columns present:",
      all(c in graph_roads.columns for c in cost_columns))

Graph roads: 4028
Dynamic cost columns present: True


In [107]:
# Use the graph road table directly

graph_roads = graph_roads.reset_index(drop=True)

for idx, row in graph_roads.iterrows():

    # Find the corresponding graph edge
    matches = [
        (u, v, data)
        for u, v, data in road_graph.edges(data=True)
        if data["road_index"] == idx
    ]

    for _, _, data in matches:
        for t in [48, 42, 36, 30, 24, 18, 12, 6, 0]:
            data[f"cost_{t}h"] = float(
                row[f"routing_cost_Tminus_{t}h"]
            )

print("Dynamic routing costs attached.")

Dynamic routing costs attached.


In [108]:
sample_edges = list(road_graph.edges(data=True))[:3]

for u, v, data in sample_edges:
    print(
        "Edge:",
        round(data["length_km"], 3), "km |",
        "T-48:", round(data["cost_48h"], 3), "|",
        "T-24:", round(data["cost_24h"], 3), "|",
        "T-0:", round(data["cost_0h"], 3)
    )

Edge: 64.092 km | T-48: 87.298 | T-24: 108.847 | T-0: 121.555
Edge: 65.256 km | T-48: 117.236 | T-24: 165.503 | T-0: 193.968
Edge: 109.332 km | T-48: 139.681 | T-24: 167.862 | T-0: 184.482


In [109]:
# Find the largest connected road component
components = list(nx.connected_components(road_graph))
largest_component = max(components, key=len)

routing_graph = road_graph.subgraph(
    largest_component
).copy()

print("Routing graph nodes:", routing_graph.number_of_nodes())
print("Routing graph edges:", routing_graph.number_of_edges())

# Pick two nodes reasonably far apart
nodes = list(routing_graph.nodes)

start_node = nodes[0]
end_node = nodes[-1]

print("\nStart:", start_node)
print("End:", end_node)

Routing graph nodes: 795
Routing graph edges: 962

Start: (9829809.582793236, 2567846.0444800537)
End: (9827535.147370828, 2569695.052155995)


In [110]:
route_results = {}

for t in [48, 24, 12, 6, 0]:

    try:
        path = nx.shortest_path(
            routing_graph,
            source=start_node,
            target=end_node,
            weight=f"cost_{t}h"
        )

        route_cost = nx.shortest_path_length(
            routing_graph,
            source=start_node,
            target=end_node,
            weight=f"cost_{t}h"
        )

        route_results[t] = {
            "path": path,
            "cost": route_cost,
            "nodes": len(path)
        }

        print(
            f"T-{t}h → "
            f"cost={route_cost:.2f}, "
            f"nodes={len(path)}"
        )

    except nx.NetworkXNoPath:
        print(f"T-{t}h → No route")

T-48h → cost=14.23, nodes=57
T-24h → cost=20.66, nodes=57
T-12h → cost=24.05, nodes=57
T-6h → cost=24.62, nodes=65
T-0h → cost=24.46, nodes=57


In [111]:
print("=== ROUTE DEGRADATION ===")

if 48 in route_results:

    baseline_cost = route_results[48]["cost"]

    for t, result in route_results.items():

        change = (
            (result["cost"] - baseline_cost)
            / baseline_cost
        ) * 100

        print(
            f"T-{t}h: "
            f"{result['cost']:.2f} "
            f"({change:+.2f}% vs T-48h)"
        )

=== ROUTE DEGRADATION ===
T-48h: 14.23 (+0.00% vs T-48h)
T-24h: 20.66 (+45.20% vs T-48h)
T-12h: 24.05 (+68.96% vs T-48h)
T-6h: 24.62 (+73.00% vs T-48h)
T-0h: 24.46 (+71.85% vs T-48h)


In [112]:
route_summary = []

for t, result in route_results.items():

    route_summary.append({
        "time_to_landfall_h": t,
        "routing_cost": round(result["cost"], 4),
        "route_nodes": result["nodes"],
        "route_change_percent": round(
            ((result["cost"] - route_results[48]["cost"])
             / route_results[48]["cost"]) * 100,
            2
        )
    })

route_summary_df = pd.DataFrame(route_summary)

print(route_summary_df)

   time_to_landfall_h  routing_cost  route_nodes  route_change_percent
0                  48       14.2315           57                  0.00
1                  24       20.6640           57                 45.20
2                  12       24.0451           57                 68.96
3                   6       24.6211           65                 73.00
4                   0       24.4574           57                 71.85


In [113]:
route_coordinates = {}

for t, result in route_results.items():

    route_coordinates[t] = [
        {
            "x": round(float(node[0]), 2),
            "y": round(float(node[1]), 2)
        }
        for node in result["path"]
    ]

print("Route coordinate sets:", list(route_coordinates.keys()))
print("T-48h points:", len(route_coordinates[48]))

Route coordinate sets: [48, 24, 12, 6, 0]
T-48h points: 57


In [114]:
import json

routing_output = {
    "start_node": {
        "x": float(start_node[0]),
        "y": float(start_node[1])
    },
    "end_node": {
        "x": float(end_node[0]),
        "y": float(end_node[1])
    },
    "route_summary": route_summary,
    "routes": {
        str(t): coords
        for t, coords in route_coordinates.items()
    }
}

with open("/content/cyclone_routing_output.json", "w") as f:
    json.dump(routing_output, f, indent=2)

print("Saved: /content/cyclone_routing_output.json")

Saved: /content/cyclone_routing_output.json


In [115]:
# Clean frontend/API-ready routing payload

routing_api_payload = {
    "start": {
        "x": round(float(start_node[0]), 2),
        "y": round(float(start_node[1]), 2)
    },
    "end": {
        "x": round(float(end_node[0]), 2),
        "y": round(float(end_node[1]), 2)
    },
    "routes": {}
}

for t, result in route_results.items():

    routing_api_payload["routes"][str(t)] = {
        "time_to_landfall_h": t,
        "routing_cost": round(float(result["cost"]), 4),
        "route_nodes": int(result["nodes"]),
        "change_percent": round(
            (
                (result["cost"] - route_results[48]["cost"])
                / route_results[48]["cost"]
            ) * 100,
            2
        ),
        "path": [
            [round(float(node[0]), 2), round(float(node[1]), 2)]
            for node in result["path"]
        ]
    }

print("Routing API payload ready.")
print("Times:", list(routing_api_payload["routes"].keys()))

Routing API payload ready.
Times: ['48', '24', '12', '6', '0']


In [116]:
with open("/content/routing_api_payload.json", "w") as f:
    json.dump(routing_api_payload, f, indent=2)

print("Saved: /content/routing_api_payload.json")

Saved: /content/routing_api_payload.json


In [117]:
for t, route in routing_api_payload["routes"].items():

    print(
        f"T-{t}h | "
        f"cost={route['routing_cost']} | "
        f"change={route['change_percent']}% | "
        f"points={len(route['path'])}"
    )

T-48h | cost=14.2315 | change=0.0% | points=57
T-24h | cost=20.664 | change=45.2% | points=57
T-12h | cost=24.0451 | change=68.96% | points=57
T-6h | cost=24.6211 | change=73.0% | points=65
T-0h | cost=24.4574 | change=71.85% | points=57


In [118]:
import json

with open("/content/routing_api_payload.json", "r") as f:
    routing_api_payload = json.load(f)

print("Routing payload loaded.")
print("Available forecast times:",
      list(routing_api_payload["routes"].keys()))

Routing payload loaded.
Available forecast times: ['48', '24', '12', '6', '0']


In [119]:
@app.get("/routing")
def get_routing():
    return routing_api_payload

NameError: name 'app' is not defined

In [120]:
import json

with open("/content/routing_api_payload.json", "r") as f:
    routing_api_payload = json.load(f)

print("Routing payload loaded.")
print("Routes:", list(routing_api_payload["routes"].keys()))
print("FastAPI app exists:", "app" in globals())


Routing payload loaded.
Routes: ['48', '24', '12', '6', '0']
FastAPI app exists: False


In [121]:
@app.get("/routing")
def get_routing():
    return routing_api_payload

print("Routing endpoint added.")

NameError: name 'app' is not defined

In [123]:
from google.colab import files

files.download("/content/routing_api_payload.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [124]:
from pyproj import Transformer

transformer = Transformer.from_crs(
    "EPSG:3857",
    "EPSG:4326",
    always_xy=True
)

routing_api_payload_wgs84 = {
    "start": {},
    "end": {},
    "routes": {}
}

# Convert start
start_lon, start_lat = transformer.transform(
    routing_api_payload["start"]["x"],
    routing_api_payload["start"]["y"]
)

routing_api_payload_wgs84["start"] = {
    "latitude": round(start_lat, 6),
    "longitude": round(start_lon, 6)
}

# Convert end
end_lon, end_lat = transformer.transform(
    routing_api_payload["end"]["x"],
    routing_api_payload["end"]["y"]
)

routing_api_payload_wgs84["end"] = {
    "latitude": round(end_lat, 6),
    "longitude": round(end_lon, 6)
}

# Convert every route point
for t, route in routing_api_payload["routes"].items():

    converted_path = []

    for x, y in route["path"]:
        lon, lat = transformer.transform(x, y)

        converted_path.append([
            round(lat, 6),
            round(lon, 6)
        ])

    routing_api_payload_wgs84["routes"][t] = {
        "time_to_landfall_h": route["time_to_landfall_h"],
        "routing_cost": route["routing_cost"],
        "route_nodes": route["route_nodes"],
        "change_percent": route["change_percent"],
        "path": converted_path
    }

print("Coordinate conversion complete.")

Coordinate conversion complete.


In [125]:
print("Start:", routing_api_payload_wgs84["start"])
print("End:", routing_api_payload_wgs84["end"])

for t in ["48", "24", "12", "6", "0"]:
    path = routing_api_payload_wgs84["routes"][t]["path"]

    print(
        f"T-{t}h → "
        f"points={len(path)}, "
        f"first={path[0]}, "
        f"last={path[-1]}"
    )

Start: {'latitude': 22.468318, 'longitude': 88.302682}
End: {'latitude': 22.483666, 'longitude': 88.28225}
T-48h → points=57, first=[22.468318, 88.302682], last=[22.483666, 88.28225]
T-24h → points=57, first=[22.468318, 88.302682], last=[22.483666, 88.28225]
T-12h → points=57, first=[22.468318, 88.302682], last=[22.483666, 88.28225]
T-6h → points=65, first=[22.468318, 88.302682], last=[22.483666, 88.28225]
T-0h → points=57, first=[22.468318, 88.302682], last=[22.483666, 88.28225]


In [126]:
with open("/content/routing_api_payload_wgs84.json", "w") as f:
    json.dump(routing_api_payload_wgs84, f, indent=2)

print("Saved: routing_api_payload_wgs84.json")

Saved: routing_api_payload_wgs84.json


In [127]:
from google.colab import files

files.download("/content/routing_api_payload_wgs84.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import os

print(os.listdir("/content"))

['.config', 'sample_data']


In [4]:
from google.colab import files

uploaded = files.upload()

Saving routing_api_payload_wgs84.json to routing_api_payload_wgs84 (1).json


In [5]:
import json

with open("/content/routing_api_payload_wgs84.json", "r") as f:
    routing_api_payload_wgs84 = json.load(f)

print("WGS84 routing payload loaded.")
print("Routes:", list(routing_api_payload_wgs84["routes"].keys()))

WGS84 routing payload loaded.
Routes: ['48', '24', '12', '6', '0']


In [6]:
required_times = ["48", "24", "12", "6", "0"]

assert set(routing_api_payload_wgs84["routes"].keys()) == set(required_times)

for t in required_times:
    route = routing_api_payload_wgs84["routes"][t]

    assert len(route["path"]) > 0

    lat, lon = route["path"][0]

    assert -90 <= lat <= 90
    assert -180 <= lon <= 180

print("✓ All 5 routes present")
print("✓ All route paths non-empty")
print("✓ Coordinates verified as latitude/longitude")
print("✓ Routing artifact is ready for GitHub")

✓ All 5 routes present
✓ All route paths non-empty
✓ Coordinates verified as latitude/longitude
✓ Routing artifact is ready for GitHub
